In [1]:
!pip install -q -U \
    "transformers>=4.53.0" \
    "datasets>=3.6.0" \
    "accelerate>=1.8.0" \
    "peft>=0.15.0" \
    pandas \
    openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 84.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 89.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.5 which is incompatible.
google-colab 1.0.0 requires jupyter-serve

In [2]:
# ============================================================
# IMPORTS AND GENERAL CONFIG
# ============================================================

import gc
import os
import re
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 250)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
# ============================================================
# PIPELINE CONFIG
# ============================================================

DATASET_ID = "TuwaiqAcademy/AISA-ArabicFC"

# حاليًا نستخدم dev كأنه test
USE_DEV_AS_TEST = False

DEV_SPLIT_CANDIDATES = [
    "dev",
    "validation",
    "valid",
]

# عندما ينزل test الرسمي نضيف أسماءه هنا
TEST_SPLIT_CANDIDATES = [
    "test",
    "blind_test",
    "blind",
]

TOOL_MODEL_REPO = "SabahBa67/ArgTune-Tool-Selector"
TOOL_MODEL_SUBFOLDER = "Tool-Classifir-merged"


# أثناء التجربة السريعة ضعي رقمًا مثل 20
# ولتشغيل كامل dev ضعي None
TEST_LIMIT = None

MAX_INPUT_LENGTH = 2048
MAX_NEW_TOKENS = 256
TOOL_BATCH_SIZE = 4

BASE_OUTPUT_DIR = Path("outputs")

PIPELINE_OUTPUT_DIR = (
    BASE_OUTPUT_DIR
    / "aisa_final_pipeline"
)

PIPELINE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TOOL_RESULTS_CSV = (
    PIPELINE_OUTPUT_DIR
    / "tool_predictions.csv"
)

TOOL_RESULTS_JSONL = (
    PIPELINE_OUTPUT_DIR
    / "tool_predictions.jsonl"
)

ARGUMENT_INPUT_CSV = (
    PIPELINE_OUTPUT_DIR
    / "argument_models_input.csv"
)

ARGUMENT_INPUT_JSONL = (
    PIPELINE_OUTPUT_DIR
    / "argument_models_input.jsonl"
)

print("Output directory:", PIPELINE_OUTPUT_DIR)
print("Tool model:", TOOL_MODEL_REPO)

Output directory: /kaggle/working/aisa_final_pipeline
Tool model: /kaggle/input/datasets/sabahbaothman/tool-classifir-model


In [5]:
# ============================================================
# LOAD DATASET
# ============================================================

dataset = load_dataset(DATASET_ID)

print(dataset)
print("Available splits:", list(dataset.keys()))

if USE_DEV_AS_TEST:
    selected_split = next(
        (
            split_name
            for split_name in DEV_SPLIT_CANDIDATES
            if split_name in dataset
        ),
        None,
    )
else:
    selected_split = next(
        (
            split_name
            for split_name in TEST_SPLIT_CANDIDATES
            if split_name in dataset
        ),
        None,
    )

if selected_split is None:
    raise KeyError(
        "Requested split was not found. "
        f"Available splits: {list(dataset.keys())}"
    )

test_raw = [
    dict(row)
    for row in dataset[selected_split]
]

if TEST_LIMIT is not None:
    test_raw = test_raw[:TEST_LIMIT]

print("Selected split:", selected_split)
print("Rows:", len(test_raw))
print("Columns:", dataset[selected_split].column_names)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/10.6M [00:00<?, ?B/s]

data/dev-00000-of-00001.parquet:   0%|          | 0.00/678k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/727k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10550 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/545 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1125 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'requires_function', 'tool_called', 'messages', 'tools', 'tools_sampled', 'negative_category', 'dialect'],
        num_rows: 10550
    })
    dev: Dataset({
        features: ['text', 'requires_function', 'tool_called', 'messages', 'tools', 'tools_sampled', 'negative_category', 'dialect'],
        num_rows: 545
    })
    test: Dataset({
        features: ['text', 'requires_function', 'tool_called', 'messages', 'tools', 'tools_sampled', 'negative_category', 'dialect'],
        num_rows: 1125
    })
})
Available splits: ['train', 'dev', 'test']
Selected split: dev
Rows: 545
Columns: ['text', 'requires_function', 'tool_called', 'messages', 'tools', 'tools_sampled', 'negative_category', 'dialect']


In [6]:
# ============================================================
# DATA PREPARATION HELPERS
# ============================================================

def extract_user_query(messages):
    if not isinstance(messages, list):
        return ""

    for message in messages:
        if not isinstance(message, dict):
            continue

        if message.get("role") == "user":
            content = message.get(
                "content",
                "",
            )

            if isinstance(content, str):
                return content.strip()

    return ""


def build_inference_prompt(text):
    text = str(text or "").strip()

    marker = "<start_of_turn>model"

    if marker in text:
        text = (
            text.split(marker, 1)[0].rstrip()
            + "\n<start_of_turn>model\n"
        )
    else:
        text = (
            text.rstrip()
            + "\n<start_of_turn>model\n"
        )

    if not text.startswith("<bos>"):
        text = "<bos>" + text

    return text


def get_row_id(row, fallback_id):
    for key in [
        "id",
        "idx",
        "example_id",
    ]:
        value = row.get(key)

        if value is not None:
            return value

    return fallback_id


def get_gold_tool(row):
    for key in [
        "gold_tool",
        "tool_called",
        "tool",
        "tool_name",
    ]:
        value = row.get(key)

        if (
            isinstance(value, str)
            and value.strip()
        ):
            return value.strip()

    return None

In [7]:
# ============================================================
# BUILD TEST DATAFRAME
# ============================================================

records = []

for fallback_id, row in enumerate(test_raw):
    row_id = get_row_id(
        row,
        fallback_id,
    )

    query = extract_user_query(
        row.get("messages")
    )

    original_text = row.get(
        "text",
        "",
    )

    prompt = build_inference_prompt(
        original_text
    )

    gold_tool = get_gold_tool(row)

    records.append({
        "row_number": fallback_id,
        "id": row_id,
        "query": query,
        "original_prompt": prompt,
        "gold_tool": gold_tool,
    })

test_df = pd.DataFrame(records)

print("Prepared rows:", len(test_df))
print("Columns:", test_df.columns.tolist())

display(test_df.head(3))

Prepared rows: 545
Columns: ['row_number', 'id', 'query', 'original_prompt', 'gold_tool']


,row_number,id,query,original_prompt,gold_tool
0,0,0,ممكن تشيكلي على مخالفات المرور بالرقم القومي 987654321؟,<bos><start_of_turn>developer\nعند الحاجة لاستدعاء أداة: اكتب أولا <think> reasoning قصير </think> ثم TOOL_CALL فقط. لا تقدم إجابة نهائية قبل استدعاء الأداة. إذا كان سؤال الطقس بلا يوم محدد فاعتبره اليوم (days=1).<start_function_declaration>decla...,check_traffic_violations
1,1,1,ممكن تقولي اتجاه القبلة في الجيزة؟,<bos><start_of_turn>developer\nعند الحاجة لاستدعاء أداة: اكتب أولا <think> reasoning قصير </think> ثم TOOL_CALL فقط. لا تقدم إجابة نهائية قبل استدعاء الأداة. إذا كان سؤال الطقس بلا يوم محدد فاعتبره اليوم (days=1).<start_function_declaration>decla...,get_qibla_direction
2,2,2,أريد تحويل ١٥٠٠ ريال سعودي إلى دولار أمريكي.,<bos><start_of_turn>developer\nعند الحاجة لاستدعاء أداة: اكتب أولا <think> reasoning قصير </think> ثم TOOL_CALL فقط. لا تقدم إجابة نهائية قبل استدعاء الأداة. إذا كان سؤال الطقس بلا يوم محدد فاعتبره اليوم (days=1).<start_function_declaration>decla...,convert_currency


In [8]:
# ============================================================
# CHECK PROMPTS
# ============================================================

empty_prompt_count = int(
    test_df["original_prompt"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

empty_query_count = int(
    test_df["query"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("Empty prompts:", empty_prompt_count)
print("Empty queries:", empty_query_count)

print("\nExample prompt:")
print("=" * 100)
print(
    test_df.iloc[0][
        "original_prompt"
    ][-3000:]
)
print("=" * 100)

if (
    test_df.iloc[0]["gold_tool"]
    is not None
):
    print(
        "Gold tool:",
        test_df.iloc[0]["gold_tool"],
    )

Empty prompts: 0
Empty queries: 0

Example prompt:
<bos><start_of_turn>developer
عند الحاجة لاستدعاء أداة: اكتب أولا <think> reasoning قصير </think> ثم TOOL_CALL فقط. لا تقدم إجابة نهائية قبل استدعاء الأداة. إذا كان سؤال الطقس بلا يوم محدد فاعتبره اليوم (days=1).<start_function_declaration>declaration:check_traffic_violations{description:<escape>الاستعلام عن المخالفات المرورية<escape>,parameters:{properties:{id_number:{description:<escape>رقم الهوية أو الإقامة<escape>,type:<escape>STRING<escape>},plate_number:{description:<escape>رقم اللوحة (اختياري)<escape>,type:<escape>STRING<escape>}},type:<escape>OBJECT<escape>}}<end_function_declaration><start_function_declaration>declaration:transfer_money{description:<escape>تحويل الأموال<escape>,parameters:{properties:{amount:{description:<escape>المبلغ<escape>,type:<escape>NUMBER<escape>},bank_name:{description:<escape>اسم البنك<escape>,type:<escape>STRING<escape>},currency:{description:<escape>العملة<escape>,type:<escape>STRING<escape>},recip

In [9]:
# ============================================================
# BUILD COMPLETE TOOL LIST
# ============================================================

if "train" not in dataset:
    raise KeyError(
        "Train split is needed to build the tool list."
    )

train_tools = set()

for row in dataset["train"]:
    tool_name = get_gold_tool(
        dict(row)
    )

    if tool_name:
        train_tools.add(tool_name)

TOOLS = sorted(train_tools)

REAL_TOOLS = [
    tool
    for tool in TOOLS
    if tool != "none"
]

print("Total labels:", len(TOOLS))
print("Real tools:", len(REAL_TOOLS))
print(TOOLS)

Total labels: 21
Real tools: 20
['book_doctor_appointment', 'calculate_customs', 'calculate_end_of_service', 'calculate_zakat', 'check_insurance_coverage', 'check_iqama_status', 'check_traffic_violations', 'check_visa_status', 'compare_prices', 'convert_currency', 'get_air_quality', 'get_qibla_direction', 'get_weather', 'none', 'order_food', 'search_hotels', 'search_medications', 'search_quran', 'search_umrah_packages', 'transfer_money', 'translate_text']


In [10]:
# ============================================================
# TOOL OUTPUT PARSER
# ============================================================

def extract_predicted_tool(
    generated_text,
):
    text = str(
        generated_text or ""
    ).strip()

    structured_patterns = [
        (
            r"<start_function_call>"
            r"\s*call\s*:\s*"
            r"([A-Za-z0-9_]+)"
        ),
        (
            r"<start_function_call>"
            r"\s*"
            r"([A-Za-z0-9_]+)"
        ),
        (
            r"<function\s*=\s*"
            r"([A-Za-z0-9_]+)>"
        ),
        (
            r'"tool_called"\s*:\s*'
            r'"([^"]+)"'
        ),
        (
            r'"name"\s*:\s*'
            r'"([^"]+)"'
        ),
        (
            r"\bcall\s*:\s*"
            r"([A-Za-z0-9_]+)"
        ),
    ]

    for pattern in structured_patterns:
        match = re.search(
            pattern,
            text,
            flags=(
                re.IGNORECASE
                | re.DOTALL
            ),
        )

        if not match:
            continue

        candidate = (
            match.group(1)
            .strip()
        )

        exact_candidate = next(
            (
                tool
                for tool in REAL_TOOLS
                if (
                    tool.lower()
                    == candidate.lower()
                )
            ),
            None,
        )

        if exact_candidate:
            return (
                exact_candidate,
                "structured",
            )

    mentioned_tools = []

    for tool in REAL_TOOLS:
        pattern = (
            rf"(?<![A-Za-z0-9_])"
            rf"{re.escape(tool)}"
            rf"(?![A-Za-z0-9_])"
        )

        if re.search(
            pattern,
            text,
            flags=re.IGNORECASE,
        ):
            mentioned_tools.append(tool)

    if len(mentioned_tools) == 1:
        return (
            mentioned_tools[0],
            "single_tool_mention",
        )

    return (
        "none",
        "no_tool_found",
    )

In [11]:
# ============================================================
# LOAD TOOL CLASSIFIER MODEL
# ============================================================

MODEL_DTYPE = (
    torch.bfloat16
    if (
        torch.cuda.is_available()
        and torch.cuda.is_bf16_supported()
    )
    else (
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    )
)

print("Model dtype:", MODEL_DTYPE)

tool_tokenizer = (
    AutoTokenizer
    .from_pretrained(
        TOOL_MODEL_REPO,
        subfolder=TOOL_MODEL_SUBFOLDER,
        trust_remote_code=True,
    )
)

if tool_tokenizer.pad_token_id is None:
    tool_tokenizer.pad_token = (
        tool_tokenizer.eos_token
    )

tool_tokenizer.padding_side = "left"

tool_model = (
    AutoModelForCausalLM
    .from_pretrained(
        TOOL_MODEL_REPO,
        subfolder=TOOL_MODEL_SUBFOLDER,
        torch_dtype=MODEL_DTYPE,
        device_map=(
            "auto"
            if torch.cuda.is_available()
            else None
        ),
        trust_remote_code=True,
    )
)

tool_model.eval()

TOOL_MODEL_DEVICE = next(
    tool_model.parameters()
).device

print("Tool model loaded.")
print("Device:", TOOL_MODEL_DEVICE)

Model dtype: torch.bfloat16


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Tool model loaded.
Device: cuda:1


In [12]:
# ============================================================
# RUN TOOL CLASSIFIER
# ============================================================

@torch.inference_mode()
def run_tool_classifier(
    input_df,
    batch_size=TOOL_BATCH_SIZE,
):
    result_records = []

    for start_index in tqdm(
        range(
            0,
            len(input_df),
            batch_size,
        ),
        desc="Extracting tools",
    ):
        batch_df = input_df.iloc[
            start_index:
            start_index + batch_size
        ]

        prompts = (
            batch_df["original_prompt"]
            .astype(str)
            .tolist()
        )

        encoded = tool_tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LENGTH,
            add_special_tokens=False,
        )

        encoded = {
            key: value.to(
                TOOL_MODEL_DEVICE
            )
            for key, value
            in encoded.items()
        }

        generated = tool_model.generate(
            **encoded,
            max_new_tokens=(
                MAX_NEW_TOKENS
            ),
            do_sample=False,
            num_beams=1,
            use_cache=True,
            pad_token_id=(
                tool_tokenizer
                .pad_token_id
            ),
            eos_token_id=(
                tool_tokenizer
                .eos_token_id
            ),
        )

        input_length = encoded[
            "input_ids"
        ].shape[1]

        generated_only = generated[
            :,
            input_length:
        ]

        decoded_outputs = (
            tool_tokenizer.batch_decode(
                generated_only,
                skip_special_tokens=False,
            )
        )

        for (
            (_, row),
            raw_output,
        ) in zip(
            batch_df.iterrows(),
            decoded_outputs,
        ):
            (
                predicted_tool,
                parser_method,
            ) = extract_predicted_tool(
                raw_output
            )

            gold_tool = row.get(
                "gold_tool"
            )

            is_correct = None

            if (
                isinstance(gold_tool, str)
                and gold_tool.strip()
            ):
                is_correct = (
                    predicted_tool
                    == gold_tool
                )

            result_records.append({
                "row_number": row[
                    "row_number"
                ],
                "id": row["id"],
                "query": row["query"],
                "original_prompt": row[
                    "original_prompt"
                ],
                "gold_tool": gold_tool,
                "predicted_tool": (
                    predicted_tool
                ),
                "parser_method": (
                    parser_method
                ),
                "correct": is_correct,
                "tool_raw_output": (
                    raw_output
                ),
            })

    return pd.DataFrame(
        result_records
    )

In [13]:
tool_test_results = run_tool_classifier(
    test_df.head(5),
    batch_size=1,
)

Extracting tools:   0%|          | 0/5 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

In [14]:
# ============================================================
# SMALL TOOL MODEL TEST
# ============================================================

tool_test_results = run_tool_classifier(
    test_df.head(5),
    batch_size=1,
)

display(
    tool_test_results[
        [
            "id",
            "query",
            "gold_tool",
            "predicted_tool",
            "parser_method",
            "correct",
            "tool_raw_output",
        ]
    ]
)

Extracting tools:   0%|          | 0/5 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

,id,query,gold_tool,predicted_tool,parser_method,correct,tool_raw_output
0,0,ممكن تشيكلي على مخالفات المرور بالرقم القومي 987654321؟,check_traffic_violations,check_traffic_violations,structured,True,"<think>\nيبدو أن المستخدم يرغب في التحقق من المخالفات المرورية باستخدام رقم الهوية. أداة ""check_traffic_violations"" هي الأنسب لهذا الغرض لأنها مصممة للتحقق من المخالفات المرورية بناءً على رقم الهوية. المدخلات المتوقعة هي id_number=987654321.\n</t..."
1,1,ممكن تقولي اتجاه القبلة في الجيزة؟,get_qibla_direction,get_qibla_direction,structured,True,<think>\nيبدو أن المستخدم يرغب في معرفة اتجاه القبلة في مدينة الجيزة. أداة get_qibla_direction هي الأنسب لأنها مصممة لتحديد اتجاه القبلة بناءً على الموقع. المدخلات المتوقعة هي city=الجيزة.\n</think>\n<start_function_call>call:get_qibla_direction{...
2,2,أريد تحويل ١٥٠٠ ريال سعودي إلى دولار أمريكي.,convert_currency,convert_currency,structured,True,<think>\nفهمت أنك ترغب في تحويل 1500 ريال سعودي إلى دولار أمريكي. أداة convert_currency هي الأنسب لهذا الغرض لأنها مصممة لتحويل العملات بشكل دقيق. المدخلات المتوقعة هي: amount=1500.0، from_currency=SAR، to_currency=USD.\n</think>\n<start_function...
3,3,ابي اشوف مخالفات المرور اللي عليّ برقم الهوية 4433221100,check_traffic_violations,check_traffic_violations,structured,True,<think>\nيبدو أن المستخدم يريد معرفة المخالفات المرورية التي يملكها برقم الهوية الخاص به. الأداة الأنسب لهذا الغرض هي check_traffic_violations، حيث توفر معلومات دقيقة عن المخالفات المرورية. المدخلات المتوقعة هي id_number=4433221100.\n</think>\n<s...
4,4,إيه اتجاه القبلة في بورتسودان؟,get_qibla_direction,get_qibla_direction,structured,True,<think>\nيبدو أن المستخدم يرغب في معرفة اتجاه القبلة في مدينة بورتسودان. أداة get_qibla_direction هي الأنسب لهذا الغرض لأنها مصممة لتحديد اتجاه القبلة بناءً على الموقع الجغرافي. المدخلات المتوقعة هي city=بورتسودان.\n</think>\n<start_function_call...


In [15]:
# ============================================================
# RUN FULL TOOL CLASSIFICATION
# ============================================================

tool_results_df = run_tool_classifier(
    test_df,
    batch_size=TOOL_BATCH_SIZE,
)

print("Completed rows:", len(tool_results_df))

display(
    tool_results_df[
        [
            "id",
            "query",
            "gold_tool",
            "predicted_tool",
            "parser_method",
            "correct",
        ]
    ].head(20)
)

Extracting tools:   0%|          | 0/137 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

Completed rows: 545


,id,query,gold_tool,predicted_tool,parser_method,correct
0,0,ممكن تشيكلي على مخالفات المرور بالرقم القومي 987654321؟,check_traffic_violations,check_traffic_violations,structured,True
1,1,ممكن تقولي اتجاه القبلة في الجيزة؟,get_qibla_direction,get_qibla_direction,structured,True
2,2,أريد تحويل ١٥٠٠ ريال سعودي إلى دولار أمريكي.,convert_currency,convert_currency,structured,True
3,3,ابي اشوف مخالفات المرور اللي عليّ برقم الهوية 4433221100,check_traffic_violations,check_traffic_violations,structured,True
4,4,إيه اتجاه القبلة في بورتسودان؟,get_qibla_direction,get_qibla_direction,structured,True
5,5,كيف بحسب زكاة المال إذا عندي ٥٠٠٠٠ ليرة سورية؟,calculate_zakat,calculate_zakat,structured,True
6,6,وين ألقى باقات عمرة لشخصين في جدة؟,search_umrah_packages,search_umrah_packages,structured,True
7,7,ممكن تترجم ده للألماني؟ أنا مشغول جدا النهاردة,translate_text,translate_text,structured,True
8,8,أريد البحث عن آية فيها كلمة الرحمة,search_quran,search_quran,structured,True
9,9,حل لي هذه المعادلة: ٢س + ٥ = ١١,none,none,no_tool_found,True


**حذف عند اخختبار التست**

In [16]:
# # ============================================================
# # TOOL ACCURACY
# # ============================================================

# has_gold = (
#     "gold_tool"
#     in tool_results_df.columns
#     and tool_results_df[
#         "gold_tool"
#     ].notna().any()
# )

# if has_gold:
#     evaluated_rows = (
#         tool_results_df[
#             tool_results_df[
#                 "gold_tool"
#             ].notna()
#         ].copy()
#     )

#     total = len(evaluated_rows)

#     correct = int(
#         evaluated_rows[
#             "correct"
#         ].fillna(False).sum()
#     )

#     wrong = total - correct

#     accuracy = (
#         correct / total
#         if total
#         else 0
#     )

#     real_extraction_failures = int(
#         (
#             evaluated_rows[
#                 "predicted_tool"
#             ].eq("none")
#             & evaluated_rows[
#                 "gold_tool"
#             ].ne("none")
#         ).sum()
#     )

#     fallback_count = int(
#         evaluated_rows[
#             "parser_method"
#         ].eq(
#             "single_tool_mention"
#         ).sum()
#     )

#     print("=" * 70)
#     print("TOOL CLASSIFICATION RESULTS")
#     print("=" * 70)
#     print("Total:", total)
#     print("Correct:", correct)
#     print("Wrong:", wrong)
#     print(f"Accuracy: {accuracy:.2%}")
#     print(
#         "Real extraction failures:",
#         real_extraction_failures,
#     )
#     print(
#         "Fallback predictions:",
#         fallback_count,
#     )
# else:
#     print(
#         "No gold labels are available. "
#         "Tool accuracy was not calculated."
#     )

In [17]:
# # ============================================================
# # SHOW TOOL ERRORS
# # ============================================================

# if has_gold:
#     tool_errors_df = (
#         tool_results_df[
#             tool_results_df[
#                 "correct"
#             ].eq(False)
#         ].copy()
#     )

#     print(
#         "Tool errors:",
#         len(tool_errors_df),
#     )

#     display(
#         tool_errors_df[
#             [
#                 "id",
#                 "query",
#                 "gold_tool",
#                 "predicted_tool",
#                 "parser_method",
#                 "tool_raw_output",
#             ]
#         ]
#     )

**حذف عند اخختبار التست**

# Stage 2: استخراج الـArguments بالمودلين

هذه المرحلة تستخدم **نفس صيغة الإدخال التي استُخدمت أثناء تدريب كل مودل**:

- أدوات الـ7 → Specialist model
- باقي الأدوات → General model
- `none` → `{}` بدون تشغيل مودل

لا توجد Safe Rules في هذه النسخة. الهدف أولًا إنتاج التوقعات، قياس الدقة على `dev`، وحفظ ملف التسليم بنفس ترتيب الصفوف الأصلي.


In [18]:
# ============================================================
# ARGUMENT STAGE CONFIG
# عدلي المسارين فقط بعد إضافة المودلين إلى Kaggle
# ============================================================

GENERAL_ARGUMENT_MODEL_REPO = "SabahBa67/ArgTune-General-Arguments"
GENERAL_ARGUMENT_MODEL_SUBFOLDER = "ArgTune-general-remaining-tool"

SPECIALIST_ARGUMENT_MODEL_REPO = "SabahBa67/ArgTune-Specialist-Arguments"
SPECIALIST_ARGUMENT_MODEL_SUBFOLDER = "ArgTune-specialist-Arguments"


ARGUMENT_MAX_INPUT_LENGTH = 1024
ARGUMENT_MAX_NEW_TOKENS = 256

# اختبار سريع قبل الرن الكامل. ضعي 0 لتخطيه.
ARGUMENT_SMOKE_ROWS_PER_MODEL = 5

FINAL_PREDICTIONS_JSONL = (
    PIPELINE_OUTPUT_DIR / "final_predictions_before_rules.jsonl"
)

FINAL_PREDICTIONS_CSV = (
    PIPELINE_OUTPUT_DIR / "final_predictions_before_rules.csv"
)

ARGUMENT_EVALUATION_CSV = (
    PIPELINE_OUTPUT_DIR / "argument_evaluation_before_rules.csv"
)

ARGUMENT_ERRORS_XLSX = (
    PIPELINE_OUTPUT_DIR / "argument_errors_before_rules.xlsx"
)

SPECIALIST_TOOLS = {
    "order_food",
    "book_doctor_appointment",
    "check_insurance_coverage",
    "compare_prices",
    "calculate_customs",
    "transfer_money",
    "calculate_end_of_service",
}

print("General model:", GENERAL_ARGUMENT_MODEL_REPO)
print("Specialist model:", SPECIALIST_ARGUMENT_MODEL_REPO)


General model path: /kaggle/input/datasets/sabahbaothman/general-remaining-tools
Specialist model path: /kaggle/input/datasets/sabahbaothman/7-tools


In [19]:
# ============================================================
# GOLD / SCHEMA HELPERS
# نبني allowed arguments من train مثل نوتبوكات التدريب
# ============================================================

from collections import defaultdict, Counter


def safe_parse_structured(value):
    if isinstance(value, (dict, list)):
        return value

    if value is None:
        return None

    if isinstance(value, str):
        value = value.strip()

        if not value:
            return None

        try:
            return json.loads(value)
        except Exception:
            return value

    return value


def is_empty_arg_value(value):
    if value is None:
        return True

    try:
        if pd.isna(value):
            return True
    except Exception:
        pass

    if isinstance(value, str):
        normalized = value.strip().lower()

        if normalized in {
            "",
            "none",
            "null",
            "nan",
            "na",
            "n/a",
        }:
            return True

    if isinstance(value, (list, dict)) and len(value) == 0:
        return True

    return False


def clean_args(args):
    if not isinstance(args, dict):
        return {}

    return {
        key: value
        for key, value in args.items()
        if not is_empty_arg_value(value)
    }


def normalize_tool_name(value):
    if value is None:
        return None

    value = str(value).strip()

    if value in TOOLS:
        return value

    for tool in REAL_TOOLS:
        if tool in value:
            return tool

    if value.lower() == "none":
        return "none"

    return None


def extract_tool_and_args_from_messages(messages):
    messages = safe_parse_structured(messages)

    if not isinstance(messages, list):
        return None, {}

    for message in messages:
        if not isinstance(message, dict):
            continue

        tool_calls = safe_parse_structured(
            message.get("tool_calls")
        )

        if not isinstance(tool_calls, list):
            continue

        for call in tool_calls:
            if not isinstance(call, dict):
                continue

            function_obj = call.get("function", call)

            if not isinstance(function_obj, dict):
                continue

            tool_name = normalize_tool_name(
                function_obj.get("name")
            )

            arguments = safe_parse_structured(
                function_obj.get("arguments", {})
            )

            if tool_name:
                return tool_name, clean_args(arguments)

    return None, {}


def get_gold_tool_and_args(row):
    # المصدر الأساسي: messages.tool_calls
    message_tool, message_args = (
        extract_tool_and_args_from_messages(
            row.get("messages")
        )
    )

    if message_tool:
        return message_tool, clean_args(message_args)

    # fallback للـtool
    tool_name = None

    for key in [
        "gold_tool",
        "tool_called",
        "tool",
        "tool_name",
    ]:
        candidate = normalize_tool_name(row.get(key))

        if candidate:
            tool_name = candidate
            break

    # fallback للـarguments لو لها عمود مستقل
    arguments = {}

    for key in [
        "arguments",
        "gold_arguments",
        "args",
    ]:
        candidate = safe_parse_structured(row.get(key))

        if isinstance(candidate, dict):
            arguments = clean_args(candidate)
            break

    if not tool_name:
        tool_name = "none"

    return tool_name, arguments


def build_allowed_args_by_tool(train_split):
    allowed = defaultdict(set)

    for dataset_row in train_split:
        row = dict(dataset_row)
        tool_name, arguments = get_gold_tool_and_args(row)

        if not tool_name or tool_name == "none":
            continue

        for argument_name in clean_args(arguments):
            allowed[tool_name].add(argument_name)

    return {
        tool_name: sorted(argument_names)
        for tool_name, argument_names in allowed.items()
    }


allowed_args_by_tool = build_allowed_args_by_tool(
    dataset["train"]
)

print("Allowed arguments by tool:")

for tool_name in sorted(allowed_args_by_tool):
    print(
        f"{tool_name:32s}",
        allowed_args_by_tool[tool_name],
    )

missing_tool_schemas = sorted(
    set(REAL_TOOLS)
    - set(allowed_args_by_tool)
)

print("\nTools without discovered arguments:", missing_tool_schemas)


Allowed arguments by tool:
book_doctor_appointment          ['city', 'date', 'doctor_name', 'specialty']
calculate_customs                ['category', 'currency', 'destination_country', 'product_value']
calculate_end_of_service         ['salary', 'termination_type', 'years_of_service']
calculate_zakat                  ['amount', 'currency', 'type']
check_insurance_coverage         ['insurance_number', 'procedure']
check_iqama_status               ['iqama_number']
check_traffic_violations         ['id_number']
check_visa_status                ['nationality', 'passport_number', 'visa_number']
compare_prices                   ['category', 'country', 'product_name']
convert_currency                 ['amount', 'from_currency', 'to_currency']
get_air_quality                  ['city', 'country']
get_qibla_direction              ['city']
get_weather                      ['city', 'days']
order_food                       ['items', 'restaurant']
search_hotels                    ['check_in', 'chec

In [20]:
# ============================================================
# ADD GOLD ARGUMENTS TO DEV/TEST DATAFRAME
# عند test الحقيقي ستبقى gold_arguments = {}
# ============================================================

gold_records = []

for row_number, raw_row in enumerate(test_raw):
    gold_tool, gold_arguments = (
        get_gold_tool_and_args(raw_row)
    )

    gold_records.append({
        "row_number": row_number,
        "gold_tool_from_messages": gold_tool,
        "gold_arguments": clean_args(gold_arguments),
    })

gold_df = pd.DataFrame(gold_records)

argument_input_df = (
    tool_results_df[
        [
            "row_number",
            "id",
            "query",
            "gold_tool",
            "predicted_tool",
            "parser_method",
            "tool_raw_output",
        ]
    ]
    .copy()
    .rename(
        columns={
            "predicted_tool": "tool_called",
        }
    )
    .merge(
        gold_df[
            [
                "row_number",
                "gold_tool_from_messages",
                "gold_arguments",
            ]
        ],
        on="row_number",
        how="left",
        validate="one_to_one",
    )
    .sort_values("row_number")
    .reset_index(drop=True)
)

# نفضّل gold المستخرج من messages لأنه يحتوي نفس مصدر arguments
argument_input_df["gold_tool"] = (
    argument_input_df[
        "gold_tool_from_messages"
    ].where(
        argument_input_df[
            "gold_tool_from_messages"
        ].notna(),
        argument_input_df["gold_tool"],
    )
)

argument_input_df.drop(
    columns=["gold_tool_from_messages"],
    inplace=True,
)


def choose_argument_model(tool_name):
    if tool_name == "none":
        return "none"

    if tool_name in SPECIALIST_TOOLS:
        return "specialist"

    return "general"


argument_input_df["argument_model"] = (
    argument_input_df["tool_called"]
    .apply(choose_argument_model)
)

print(
    argument_input_df["argument_model"]
    .value_counts(dropna=False)
)

display(
    argument_input_df[
        [
            "row_number",
            "id",
            "query",
            "gold_tool",
            "tool_called",
            "argument_model",
            "gold_arguments",
        ]
    ].head(10)
)


argument_model
general       324
specialist    176
none           45
Name: count, dtype: int64


,row_number,id,query,gold_tool,tool_called,argument_model,gold_arguments
0,0,0,ممكن تشيكلي على مخالفات المرور بالرقم القومي 987654321؟,check_traffic_violations,check_traffic_violations,general,{'id_number': '987654321'}
1,1,1,ممكن تقولي اتجاه القبلة في الجيزة؟,get_qibla_direction,get_qibla_direction,general,{'city': 'الجيزة'}
2,2,2,أريد تحويل ١٥٠٠ ريال سعودي إلى دولار أمريكي.,convert_currency,convert_currency,general,"{'amount': 1500.0, 'from_currency': 'SAR', 'to_currency': 'USD'}"
3,3,3,ابي اشوف مخالفات المرور اللي عليّ برقم الهوية 4433221100,check_traffic_violations,check_traffic_violations,general,{'id_number': '4433221100'}
4,4,4,إيه اتجاه القبلة في بورتسودان؟,get_qibla_direction,get_qibla_direction,general,{'city': 'بورتسودان'}
5,5,5,كيف بحسب زكاة المال إذا عندي ٥٠٠٠٠ ليرة سورية؟,calculate_zakat,calculate_zakat,general,"{'amount': 50000.0, 'currency': 'SYP', 'type': 'cash'}"
6,6,6,وين ألقى باقات عمرة لشخصين في جدة؟,search_umrah_packages,search_umrah_packages,general,"{'departure_city': 'جدة', 'num_persons': 2.0}"
7,7,7,ممكن تترجم ده للألماني؟ أنا مشغول جدا النهاردة,translate_text,translate_text,general,"{'target_language': 'de', 'text': 'أنا مشغول جدا النهاردة'}"
8,8,8,أريد البحث عن آية فيها كلمة الرحمة,search_quran,search_quran,general,{'query': 'الرحمة'}
9,9,9,حل لي هذه المعادلة: ٢س + ٥ = ١١,none,none,none,{}


In [21]:
# ============================================================
# PROMPTS — EXACTLY MATCH EACH MODEL'S TRAINING NOTEBOOK
# ============================================================

GENERAL_STAGE2_TEMPLATE = """You are extracting function-call arguments from an Arabic user query.

User query:
{query}

Selected tool:
{tool_name}

Allowed arguments for this tool:
{allowed_arguments}

Output rules:
- Return ONLY a valid JSON object.
- Use only the allowed argument names.
- Include an argument only if it is explicitly stated or unambiguously derivable.
- Do NOT invent default values.
- Do NOT include null, None, empty strings, or unknown values.
"""


SPECIALIST_STAGE2_TEMPLATE = """You are extracting arguments for a function call.

Arabic user query:
{query}

Selected tool:
{tool_name}

Allowed arguments:
{allowed_arguments}

Rules:
- Return ONLY a valid JSON object.
- Use only the allowed argument names.
- Do not add an argument unless it is explicitly mentioned or clearly implied.
- Do not invent default values.
- If no arguments are needed, return {{}}.
"""


def build_argument_prompt(
    tokenizer,
    query,
    tool_name,
    model_kind,
):
    allowed_arguments = json.dumps(
        allowed_args_by_tool.get(tool_name, []),
        ensure_ascii=False,
    )

    if model_kind == "specialist":
        instruction = SPECIALIST_STAGE2_TEMPLATE.format(
            query=query,
            tool_name=tool_name,
            allowed_arguments=allowed_arguments,
        )
    elif model_kind == "general":
        instruction = GENERAL_STAGE2_TEMPLATE.format(
            query=query,
            tool_name=tool_name,
            allowed_arguments=allowed_arguments,
        )
    else:
        raise ValueError(
            f"Unsupported model_kind: {model_kind}"
        )

    return tokenizer.apply_chat_template(
        [
            {
                "role": "user",
                "content": instruction,
            }
        ],
        add_generation_prompt=True,
        tokenize=False,
    )


def parse_json_arguments(text):
    raw_text = str(text or "").strip()

    cleaned_text = (
        raw_text
        .replace("```json", "")
        .replace("```JSON", "")
        .replace("```", "")
        .replace("<end_of_turn>", "")
        .replace("<eos>", "")
        .replace("<pad>", "")
        .strip()
    )

    try:
        parsed = json.loads(cleaned_text)

        if isinstance(parsed, dict):
            return parsed, False, "full_json"
    except Exception:
        pass

    json_match = re.search(
        r"\{.*\}",
        cleaned_text,
        flags=re.DOTALL,
    )

    if json_match:
        try:
            parsed = json.loads(
                json_match.group(0)
            )

            if isinstance(parsed, dict):
                return parsed, False, "json_substring"
        except Exception:
            pass

    return {}, True, "parse_failed"


def clean_prediction_args(
    predicted_arguments,
    tool_name,
):
    predicted_arguments = clean_args(
        predicted_arguments
    )

    allowed = set(
        allowed_args_by_tool.get(tool_name, [])
    )

    if not allowed:
        return {}

    return {
        key: value
        for key, value in predicted_arguments.items()
        if key in allowed
    }


In [22]:
# ============================================================
# FREE TOOL CLASSIFIER FROM GPU BEFORE LOADING ARGUMENT MODELS
# ============================================================

try:
    del tool_model
except NameError:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Tool model released from memory.")


Tool model released from memory.


In [23]:
# ============================================================
# ARGUMENT MODEL LOAD / GENERATION HELPERS
# ============================================================

def load_argument_model(model_repo, model_subfolder):
    tokenizer = AutoTokenizer.from_pretrained(
        model_repo,
        subfolder=model_subfolder,
        trust_remote_code=True,
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_repo,
        subfolder=model_subfolder,
        torch_dtype=MODEL_DTYPE,
        device_map=(
            "auto"
            if torch.cuda.is_available()
            else None
        ),
        trust_remote_code=True,
    )

    model.eval()

    return tokenizer, model


def get_specialist_eos_ids(tokenizer):
    eos_ids = set()

    if tokenizer.eos_token_id is not None:
        eos_ids.add(tokenizer.eos_token_id)

    end_turn_id = tokenizer.convert_tokens_to_ids(
        "<end_of_turn>"
    )

    if (
        isinstance(end_turn_id, int)
        and end_turn_id >= 0
        and end_turn_id != tokenizer.unk_token_id
    ):
        eos_ids.add(end_turn_id)

    return sorted(eos_ids)


@torch.inference_mode()
def generate_argument_output(
    prompt,
    tokenizer,
    model,
    model_kind,
):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=ARGUMENT_MAX_INPUT_LENGTH,
        add_special_tokens=False,
    ).to(model.device)

    if model_kind == "specialist":
        eos_ids = get_specialist_eos_ids(
            tokenizer
        )

        pad_id = tokenizer.pad_token_id

        bad_words_ids = None

        if (
            pad_id is not None
            and pad_id not in eos_ids
        ):
            bad_words_ids = [[pad_id]]

        generated = model.generate(
            **inputs,
            max_new_tokens=ARGUMENT_MAX_NEW_TOKENS,
            min_new_tokens=5,
            do_sample=False,
            pad_token_id=(
                eos_ids[0]
                if eos_ids
                else tokenizer.eos_token_id
            ),
            eos_token_id=(
                eos_ids
                if eos_ids
                else tokenizer.eos_token_id
            ),
            bad_words_ids=bad_words_ids,
            repetition_penalty=1.05,
        )

        skip_special_tokens = False

    else:
        generated = model.generate(
            **inputs,
            max_new_tokens=ARGUMENT_MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        skip_special_tokens = True

    new_tokens = generated[
        0,
        inputs["input_ids"].shape[-1]:,
    ]

    raw_output = tokenizer.decode(
        new_tokens,
        skip_special_tokens=skip_special_tokens,
    ).strip()

    return raw_output


def predict_argument_subset(
    subset_df,
    tokenizer,
    model,
    model_kind,
    description,
):
    predictions = []

    for _, row in tqdm(
        subset_df.iterrows(),
        total=len(subset_df),
        desc=description,
    ):
        tool_name = row["tool_called"]

        prompt = build_argument_prompt(
            tokenizer=tokenizer,
            query=row["query"],
            tool_name=tool_name,
            model_kind=model_kind,
        )

        raw_output = generate_argument_output(
            prompt=prompt,
            tokenizer=tokenizer,
            model=model,
            model_kind=model_kind,
        )

        (
            parsed_arguments,
            parse_failed,
            parse_method,
        ) = parse_json_arguments(raw_output)

        cleaned_arguments = clean_prediction_args(
            parsed_arguments,
            tool_name,
        )

        predictions.append({
            "row_number": row["row_number"],
            "predicted_arguments": cleaned_arguments,
            "argument_raw_output": raw_output,
            "argument_parse_failed": parse_failed,
            "argument_parse_method": parse_method,
            "argument_prompt": prompt,
        })

    return pd.DataFrame(predictions)


In [24]:
# ============================================================
# RUN GENERAL ARGUMENT MODEL
# Smoke test first, then the complete routed subset
# ============================================================

general_input_df = (
    argument_input_df[
        argument_input_df["argument_model"]
        .eq("general")
    ]
    .copy()
    .sort_values("row_number")
    .reset_index(drop=True)
)

general_tokenizer, general_model = (
    load_argument_model(
        GENERAL_ARGUMENT_MODEL_REPO,
        GENERAL_ARGUMENT_MODEL_SUBFOLDER,
    )
)


if (
    ARGUMENT_SMOKE_ROWS_PER_MODEL
    and len(general_input_df) > 0
):
    general_smoke_df = predict_argument_subset(
        subset_df=general_input_df.head(
            ARGUMENT_SMOKE_ROWS_PER_MODEL
        ),
        tokenizer=general_tokenizer,
        model=general_model,
        model_kind="general",
        description="General smoke test",
    )

    display(
        general_input_df.head(
            ARGUMENT_SMOKE_ROWS_PER_MODEL
        )[
            [
                "id",
                "query",
                "tool_called",
                "gold_arguments",
            ]
        ]
        .merge(
            general_smoke_df[
                [
                    "row_number",
                    "predicted_arguments",
                    "argument_raw_output",
                    "argument_parse_failed",
                ]
            ],
            left_index=True,
            right_index=True,
            how="left",
        )
    )

general_predictions_df = predict_argument_subset(
    subset_df=general_input_df,
    tokenizer=general_tokenizer,
    model=general_model,
    model_kind="general",
    description="General arguments",
)

print(
    "General rows completed:",
    len(general_predictions_df),
)

print(
    "General parse failures:",
    int(
        general_predictions_df[
            "argument_parse_failed"
        ].sum()
    ),
)

del general_model
del general_tokenizer

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

General smoke test:   0%|          | 0/5 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

,id,query,tool_called,gold_arguments,row_number,predicted_arguments,argument_raw_output,argument_parse_failed
0,0,ممكن تشيكلي على مخالفات المرور بالرقم القومي 987654321؟,check_traffic_violations,{'id_number': '987654321'},0,{'id_number': '987654321'},"{""id_number"": ""987654321""}",False
1,1,ممكن تقولي اتجاه القبلة في الجيزة؟,get_qibla_direction,{'city': 'الجيزة'},1,{'city': 'الجيزة'},"{""city"": ""الجيزة""}",False
2,2,أريد تحويل ١٥٠٠ ريال سعودي إلى دولار أمريكي.,convert_currency,"{'amount': 1500.0, 'from_currency': 'SAR', 'to_currency': 'USD'}",2,"{'amount': 1500.0, 'from_currency': 'SAR', 'to_currency': 'USD'}","{""amount"": 1500.0, ""from_currency"": ""SAR"", ""to_currency"": ""USD""}",False
3,3,ابي اشوف مخالفات المرور اللي عليّ برقم الهوية 4433221100,check_traffic_violations,{'id_number': '4433221100'},3,{'id_number': '4433221100'},"{""id_number"": ""4433221100""}",False
4,4,إيه اتجاه القبلة في بورتسودان؟,get_qibla_direction,{'city': 'بورتسودان'},4,{'city': 'بورتسودان'},"{""city"": ""بورتسودان""}",False


General arguments:   0%|          | 0/324 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

General rows completed: 324
General parse failures: 0


In [25]:
# ============================================================
# RUN SPECIALIST 7-TOOL ARGUMENT MODEL
# Uses the exact specialist prompt + generation settings
# ============================================================

specialist_input_df = (
    argument_input_df[
        argument_input_df["argument_model"]
        .eq("specialist")
    ]
    .copy()
    .sort_values("row_number")
    .reset_index(drop=True)
)


specialist_tokenizer, specialist_model = (
    load_argument_model(
        SPECIALIST_ARGUMENT_MODEL_REPO,
        SPECIALIST_ARGUMENT_MODEL_SUBFOLDER,
    )
)


if (
    ARGUMENT_SMOKE_ROWS_PER_MODEL
    and len(specialist_input_df) > 0
):
    specialist_smoke_df = predict_argument_subset(
        subset_df=specialist_input_df.head(
            ARGUMENT_SMOKE_ROWS_PER_MODEL
        ),
        tokenizer=specialist_tokenizer,
        model=specialist_model,
        model_kind="specialist",
        description="Specialist smoke test",
    )

    display(
        specialist_input_df.head(
            ARGUMENT_SMOKE_ROWS_PER_MODEL
        )[
            [
                "id",
                "query",
                "tool_called",
                "gold_arguments",
            ]
        ]
        .merge(
            specialist_smoke_df[
                [
                    "row_number",
                    "predicted_arguments",
                    "argument_raw_output",
                    "argument_parse_failed",
                ]
            ],
            left_index=True,
            right_index=True,
            how="left",
        )
    )

specialist_predictions_df = (
    predict_argument_subset(
        subset_df=specialist_input_df,
        tokenizer=specialist_tokenizer,
        model=specialist_model,
        model_kind="specialist",
        description="Specialist arguments",
    )
)

print(
    "Specialist rows completed:",
    len(specialist_predictions_df),
)

print(
    "Specialist parse failures:",
    int(
        specialist_predictions_df[
            "argument_parse_failed"
        ].sum()
    ),
)

del specialist_model
del specialist_tokenizer

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Specialist smoke test:   0%|          | 0/5 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

,id,query,tool_called,gold_arguments,row_number,predicted_arguments,argument_raw_output,argument_parse_failed
0,11,قارن لي أسعار هاتف iPhone 13 في مصر والسعودية,compare_prices,"{'country': 'مصر', 'product_name': 'iPhone 13'}",11,"{'country': 'مصر، السعودية', 'product_name': 'iPhone 13'}","{""country"": ""مصر، السعودية"", ""product_name"": ""iPhone 13""}<end_of_turn>",False
1,12,كم تبلغ الجمارك على حقيبة يد قيمتها 500 ريال في الإمارات؟,calculate_customs,"{'category': 'حقيبة يد', 'destination_country': 'الإمارات', 'product_value': 500.0}",12,"{'category': 'حقيبة يد', 'destination_country': 'الإمارات', 'product_value': 500.0}","{""category"": ""حقيبة يد"", ""destination_country"": ""الإمارات"", ""product_value"": 500.0}<end_of_turn>",False
2,15,أريد حجز موعد مع طبيب الأطفال في أبو ظبي يوم 15 أكتوبر,book_doctor_appointment,"{'city': 'أبو ظبي', 'date': '15 أكتوبر', 'specialty': 'طب الأطفال'}",15,"{'city': 'أبو ظبي', 'date': '15 أكتوبر', 'specialty': 'طبيب الأطفال'}","{""city"": ""أبو ظبي"", ""date"": ""15 أكتوبر"", ""specialty"": ""طبيب الأطفال""}<end_of_turn>",False
3,16,احجز لي موعدًا مع طبيب نسائية في الرياض الأسبوع القادم,book_doctor_appointment,"{'city': 'الرياض', 'date': 'الأسبوع القادم', 'specialty': 'طبيب نسائية'}",16,"{'city': 'الرياض', 'date': 'next week', 'specialty': 'نسائية'}","{""city"": ""الرياض"", ""date"": ""next week"", ""specialty"": ""نسائية""}<end_of_turn>",False
4,19,وش سعر الآيفون في الكويت؟,compare_prices,"{'country': 'الكويت', 'product_name': 'iPhone'}",19,"{'country': 'الكويت', 'product_name': 'آيفون'}","{""country"": ""الكويت"", ""product_name"": ""آيفون""}<end_of_turn>",False


Specialist arguments:   0%|          | 0/176 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

Specialist rows completed: 176
Specialist parse failures: 0


In [26]:
# ============================================================
# MERGE BOTH MODEL OUTPUTS + NONE ROWS
# Preserve the exact original dev/test order
# ============================================================

none_predictions_df = (
    argument_input_df[
        argument_input_df["argument_model"]
        .eq("none")
    ][["row_number"]]
    .copy()
)

none_predictions_df["predicted_arguments"] = [
    {} for _ in range(len(none_predictions_df))
]

none_predictions_df["argument_raw_output"] = ""
none_predictions_df["argument_parse_failed"] = False
none_predictions_df["argument_parse_method"] = "none"
none_predictions_df["argument_prompt"] = ""

all_argument_predictions_df = pd.concat(
    [
        general_predictions_df,
        specialist_predictions_df,
        none_predictions_df,
    ],
    ignore_index=True,
)

assert (
    all_argument_predictions_df[
        "row_number"
    ].duplicated().sum()
    == 0
)

assert len(all_argument_predictions_df) == len(
    argument_input_df
)

final_results_df = (
    argument_input_df
    .merge(
        all_argument_predictions_df,
        on="row_number",
        how="left",
        validate="one_to_one",
    )
    .sort_values("row_number")
    .reset_index(drop=True)
)

expected_order = list(range(len(test_raw)))

assert (
    final_results_df["row_number"].tolist()
    == expected_order
), "Output order differs from the original split."

assert final_results_df["id"].tolist() == (
    test_df
    .sort_values("row_number")["id"]
    .tolist()
), "Output IDs differ from the original split."

assert final_results_df[
    "predicted_arguments"
].apply(lambda value: isinstance(value, dict)).all()

print("Merged rows:", len(final_results_df))
print("Order check: PASSED")
print("ID check: PASSED")

display(
    final_results_df[
        [
            "row_number",
            "id",
            "query",
            "gold_tool",
            "tool_called",
            "argument_model",
            "gold_arguments",
            "predicted_arguments",
            "argument_parse_failed",
        ]
    ].head(20)
)


Merged rows: 545
Order check: PASSED
ID check: PASSED


,row_number,id,query,gold_tool,tool_called,argument_model,gold_arguments,predicted_arguments,argument_parse_failed
0,0,0,ممكن تشيكلي على مخالفات المرور بالرقم القومي 987654321؟,check_traffic_violations,check_traffic_violations,general,{'id_number': '987654321'},{'id_number': '987654321'},False
1,1,1,ممكن تقولي اتجاه القبلة في الجيزة؟,get_qibla_direction,get_qibla_direction,general,{'city': 'الجيزة'},{'city': 'الجيزة'},False
2,2,2,أريد تحويل ١٥٠٠ ريال سعودي إلى دولار أمريكي.,convert_currency,convert_currency,general,"{'amount': 1500.0, 'from_currency': 'SAR', 'to_currency': 'USD'}","{'amount': 1500.0, 'from_currency': 'SAR', 'to_currency': 'USD'}",False
3,3,3,ابي اشوف مخالفات المرور اللي عليّ برقم الهوية 4433221100,check_traffic_violations,check_traffic_violations,general,{'id_number': '4433221100'},{'id_number': '4433221100'},False
4,4,4,إيه اتجاه القبلة في بورتسودان؟,get_qibla_direction,get_qibla_direction,general,{'city': 'بورتسودان'},{'city': 'بورتسودان'},False
5,5,5,كيف بحسب زكاة المال إذا عندي ٥٠٠٠٠ ليرة سورية؟,calculate_zakat,calculate_zakat,general,"{'amount': 50000.0, 'currency': 'SYP', 'type': 'cash'}","{'amount': 50000.0, 'currency': 'SYP', 'type': 'cash'}",False
6,6,6,وين ألقى باقات عمرة لشخصين في جدة؟,search_umrah_packages,search_umrah_packages,general,"{'departure_city': 'جدة', 'num_persons': 2.0}","{'departure_city': 'جدة', 'num_persons': 2.0}",False
7,7,7,ممكن تترجم ده للألماني؟ أنا مشغول جدا النهاردة,translate_text,translate_text,general,"{'target_language': 'de', 'text': 'أنا مشغول جدا النهاردة'}","{'target_language': 'de', 'text': 'أنا مشغول جدا النهاردة'}",False
8,8,8,أريد البحث عن آية فيها كلمة الرحمة,search_quran,search_quran,general,{'query': 'الرحمة'},{'query': 'الرحمة'},False
9,9,9,حل لي هذه المعادلة: ٢س + ٥ = ١١,none,none,none,{},{},False


In [27]:
# # ============================================================
# # DEV ACCURACY — DELETE / SKIP FOR BLIND TEST
# # ============================================================

# def canonicalize_json_value(value):
#     if isinstance(value, dict):
#         return {
#             key: canonicalize_json_value(value[key])
#             for key in sorted(value)
#         }

#     if isinstance(value, list):
#         return [
#             canonicalize_json_value(item)
#             for item in value
#         ]

#     return value


# def arguments_exact_match(gold_args, predicted_args):
#     return (
#         canonicalize_json_value(
#             clean_args(gold_args)
#         )
#         ==
#         canonicalize_json_value(
#             clean_args(predicted_args)
#         )
#     )


# has_gold_arguments = (
#     USE_DEV_AS_TEST
#     and final_results_df["gold_tool"].notna().any()
# )

# if has_gold_arguments:
#     final_results_df["tool_correct"] = (
#         final_results_df["tool_called"]
#         == final_results_df["gold_tool"]
#     )

#     final_results_df["arguments_correct"] = [
#         arguments_exact_match(gold, pred)
#         for gold, pred in zip(
#             final_results_df["gold_arguments"],
#             final_results_df[
#                 "predicted_arguments"
#             ],
#         )
#     ]

#     final_results_df["full_call_correct"] = (
#         final_results_df["tool_correct"]
#         & final_results_df["arguments_correct"]
#     )

#     total_rows = len(final_results_df)

#     tool_correct_count = int(
#         final_results_df["tool_correct"].sum()
#     )

    # argument_correct_count = int(
    #     final_results_df[
    #         "arguments_correct"
    #     ].sum()
    # )

    # full_correct_count = int(
    #     final_results_df[
    #         "full_call_correct"
    #     ].sum()
    # )

    # tool_correct_function_rows = (
    #     final_results_df[
    #         final_results_df["tool_correct"]
    #         & final_results_df["gold_tool"].ne("none")
    #     ]
    # )

    # conditional_argument_correct = int(
    #     tool_correct_function_rows[
    #         "arguments_correct"
    #     ].sum()
    # )

    # conditional_argument_total = len(
    #     tool_correct_function_rows
    # )

    # parse_failures = int(
    #     final_results_df[
    #         "argument_parse_failed"
    #     ].sum()
    # )

    # print("=" * 72)
    # print("END-TO-END DEV RESULTS — BEFORE RULES")
    # print("=" * 72)
    # print(
    #     f"Tool accuracy: "
    #     f"{tool_correct_count}/{total_rows} "
    #     f"= {tool_correct_count / total_rows:.2%}"
    # )
    # print(
    #     f"Arguments exact match (all rows): "
    #     f"{argument_correct_count}/{total_rows} "
    #     f"= {argument_correct_count / total_rows:.2%}"
    # )
    # print(
    #     f"Arguments exact match when tool is correct "
    #     f"(function rows only): "
    #     f"{conditional_argument_correct}/"
    #     f"{conditional_argument_total} "
    #     f"= "
    #     f"{conditional_argument_correct / conditional_argument_total:.2%}"
    #     if conditional_argument_total
    #     else "No correctly routed function rows."
    # )
    # print(
    #     f"Full function-call exact match: "
    #     f"{full_correct_count}/{total_rows} "
    #     f"= {full_correct_count / total_rows:.2%}"
    # )
    # print("Argument parse failures:", parse_failures)

    # per_model_accuracy_df = (
    #     final_results_df
    #     .groupby(
    #         "argument_model",
    #         dropna=False,
    #     )
    #     .agg(
    #         rows=("id", "size"),
    #         tool_correct=("tool_correct", "sum"),
    #         arguments_correct=(
    #             "arguments_correct",
    #             "sum",
    #         ),
    #         full_call_correct=(
    #             "full_call_correct",
    #             "sum",
    #         ),
    #         parse_failures=(
    #             "argument_parse_failed",
    #             "sum",
    #         ),
    #     )
    #     .reset_index()
    # )

    # per_model_accuracy_df[
    #     "argument_accuracy"
    # ] = (
    #     per_model_accuracy_df[
    #         "arguments_correct"
    #     ]
    #     / per_model_accuracy_df["rows"]
    # )

    # per_model_accuracy_df[
    #     "full_call_accuracy"
    # ] = (
#         per_model_accuracy_df[
#             "full_call_correct"
#         ]
#         / per_model_accuracy_df["rows"]
#     )

#     display(per_model_accuracy_df)

# else:
#     print(
#         "Blind test detected: accuracy was not calculated."
#     )


In [28]:
# # ============================================================
# # ERROR ANALYSIS FOR DEV
# # ============================================================

# if has_gold_arguments:
#     error_rows = []

#     for _, row in final_results_df.iterrows():
#         gold_args = clean_args(
#             row["gold_arguments"]
#         )

#         predicted_args = clean_args(
#             row["predicted_arguments"]
#         )

#         missing_keys = sorted(
#             set(gold_args)
#             - set(predicted_args)
#         )

#         extra_keys = sorted(
#             set(predicted_args)
#             - set(gold_args)
#         )

#         wrong_keys = sorted(
#             key
#             for key in (
#                 set(gold_args)
#                 & set(predicted_args)
#             )
#             if canonicalize_json_value(
#                 gold_args[key]
#             )
#             != canonicalize_json_value(
#                 predicted_args[key]
#             )
#         )

#         if row["full_call_correct"]:
        #     continue

        # error_rows.append({
        #     "row_number": row["row_number"],
        #     "id": row["id"],
        #     "query": row["query"],
        #     "gold_tool": row["gold_tool"],
        #     "predicted_tool": row["tool_called"],
        #     "argument_model": row["argument_model"],
        #     "gold_arguments": json.dumps(
        #         gold_args,
        #         ensure_ascii=False,
        #     ),
        #     "predicted_arguments": json.dumps(
        #         predicted_args,
        #         ensure_ascii=False,
        #     ),
        #     "tool_correct": row["tool_correct"],
        #     "arguments_correct": row[
        #         "arguments_correct"
        #     ],
        #     "missing_keys": json.dumps(
        #         missing_keys,
        #         ensure_ascii=False,
        #     ),
        #     "extra_keys": json.dumps(
        #         extra_keys,
        #         ensure_ascii=False,
        #     ),
        #     "wrong_keys": json.dumps(
        #         wrong_keys,
        #         ensure_ascii=False,
        #     ),
        #     "argument_parse_failed": row[
        #         "argument_parse_failed"
        #     ],
        #     "argument_raw_output": row[
        #         "argument_raw_output"
        #     ],
        #     "tool_raw_output": row[
        #         "tool_raw_output"
        #     ],
        # })

    # argument_errors_df = pd.DataFrame(
    #     error_rows
    # )

    # print("Full-call errors:", len(argument_errors_df))

    # display(argument_errors_df.head(30))

    # evaluation_export_df = (
    #     final_results_df.copy()
    # )

    # for column in [
    #     "gold_arguments",
    #     "predicted_arguments",
    # ]:
    #     evaluation_export_df[column] = (
    #         evaluation_export_df[column]
    #         .apply(
    #             lambda value: json.dumps(
    #                 value,
    #                 ensure_ascii=False,
    #             )
    #         )
    #     )

    # evaluation_export_df.to_csv(
    #     ARGUMENT_EVALUATION_CSV,
    #     index=False,
    #     encoding="utf-8-sig",
    # )

    # with pd.ExcelWriter(
    #     ARGUMENT_ERRORS_XLSX,
    #     engine="openpyxl",
    # ) as writer:
    #     evaluation_export_df.to_excel(
    #         writer,
    #         sheet_name="all_rows",
    #         index=False,
    #     )

    #     argument_errors_df.to_excel(
    #         writer,
    #         sheet_name="errors",
    #         index=False,
    #     )

    #     per_model_accuracy_df.to_excel(
    #         writer,
    #         sheet_name="summary_by_model",
    #         index=False,
    #     )

    # print("Saved evaluation:")
    # print("-", ARGUMENT_EVALUATION_CSV)
    # print("-", ARGUMENT_ERRORS_XLSX)


In [29]:
# ============================================================
# SAVE FINAL SUBMISSION-LIKE FILE
# Exact original order and IDs — BEFORE SAFE RULES
# ============================================================

submission_records = []

for _, row in final_results_df.iterrows():
    submission_records.append({
        "id": row["id"],
        "tool_called": row["tool_called"],
        "arguments": clean_args(
            row["predicted_arguments"]
        ),
    })

assert len(submission_records) == len(test_raw)

assert [
    record["id"]
    for record in submission_records
] == test_df.sort_values(
    "row_number"
)["id"].tolist()

with open(
    FINAL_PREDICTIONS_JSONL,
    "w",
    encoding="utf-8",
) as output_file:
    for record in submission_records:
        output_file.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )

submission_csv_df = pd.DataFrame(
    {
        "id": [
            record["id"]
            for record in submission_records
        ],
        "tool_called": [
            record["tool_called"]
            for record in submission_records
        ],
        "arguments": [
            json.dumps(
                record["arguments"],
                ensure_ascii=False,
            )
            for record in submission_records
        ],
    }
)

submission_csv_df.to_csv(
    FINAL_PREDICTIONS_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("Final files saved:")
print("-", FINAL_PREDICTIONS_JSONL)
print("-", FINAL_PREDICTIONS_CSV)

print("\nFirst 5 JSONL records:")

for record in submission_records[:5]:
    print(
        json.dumps(
            record,
            ensure_ascii=False,
        )
    )


Final files saved:
- /kaggle/working/aisa_final_pipeline/final_predictions_before_rules.jsonl
- /kaggle/working/aisa_final_pipeline/final_predictions_before_rules.csv

First 5 JSONL records:
{"id": 0, "tool_called": "check_traffic_violations", "arguments": {"id_number": "987654321"}}
{"id": 1, "tool_called": "get_qibla_direction", "arguments": {"city": "الجيزة"}}
{"id": 2, "tool_called": "convert_currency", "arguments": {"amount": 1500.0, "from_currency": "SAR", "to_currency": "USD"}}
{"id": 3, "tool_called": "check_traffic_violations", "arguments": {"id_number": "4433221100"}}
{"id": 4, "tool_called": "get_qibla_direction", "arguments": {"city": "بورتسودان"}}


In [30]:
# ============================================================
# FINAL PIPELINE VALIDATION
# ============================================================

with open(
    FINAL_PREDICTIONS_JSONL,
    "r",
    encoding="utf-8",
) as input_file:
    saved_records = [
        json.loads(line)
        for line in input_file
        if line.strip()
    ]

assert len(saved_records) == len(test_raw)

assert [
    record["id"]
    for record in saved_records
] == test_df.sort_values(
    "row_number"
)["id"].tolist()

assert all(
    set(record) == {
        "id",
        "tool_called",
        "arguments",
    }
    for record in saved_records
)

assert all(
    isinstance(record["arguments"], dict)
    for record in saved_records
)

assert all(
    record["tool_called"] in TOOLS
    for record in saved_records
)

print("=" * 72)
print("FINAL VALIDATION PASSED")
print("=" * 72)
print("Split:", selected_split)
print("Rows:", len(saved_records))
print(
    "Unique IDs:",
    len({record["id"] for record in saved_records}),
)
print(
    "Argument parse failures:",
    int(
        final_results_df[
            "argument_parse_failed"
        ].sum()
    ),
)
print("Output:", FINAL_PREDICTIONS_JSONL)


FINAL VALIDATION PASSED
Split: dev
Rows: 545
Unique IDs: 545
Argument parse failures: 0
Output: /kaggle/working/aisa_final_pipeline/final_predictions_before_rules.jsonl


# Stage 3: Safe Rules — General + Specialist

شغّلي هذه الخلايا بعد اكتمال `final_results_df`.


In [31]:
# ---------------------------------------------------------------------------
# Safe post-processing rules for LoRA raw predictions
# Best selected rule: hotel_iban only
# No weather rule because get_weather.days is inconsistent in dev gold
# ---------------------------------------------------------------------------

import re
import copy

# RULED_PRED_PATH = f"{EXP_DIR}/lora_stage2_predictions_raw_hotel_iban.jsonl"
# RULED_ERROR_ANALYSIS_XLSX_PATH = f"{EXP_DIR}/lora_stage2_error_analysis_hotel_iban.xlsx"


def _to_text(x):
    return "" if x is None else str(x)


def query_mentions_guest_count(query):
    q = _to_text(query)

    q = q.replace("أشخاص", "اشخاص")
    q = q.replace("أفراد", "افراد")
    q = q.replace("لـ", "ل")

    # أي رقم قريب من كلمة ضيف/شخص/نفر/فرد
    if re.search(r"(\d+|[٠-٩]+)\s*(ضيف|ضيوف|شخص|اشخاص|نفر|فرد|افراد)", q):
        return True

    # أي كلمة واضحة تدل إن المستخدم ذكر أشخاص/ضيوف
    guest_words = [
        "شخص واحد",
        "لشخص واحد",
        "ل شخص واحد",
        "لشخص",
        "ل شخص",
        "شخصين",
        "لشخصين",
        "ل شخصين",
        "شخصان",
        "اشخاص",
        "ضيف واحد",
        "لضيف واحد",
        "ل ضيف واحد",
        "ضيفين",
        "ضيفان",
        "ضيوف",
        "نفر",
        "فرد",
        "افراد",
    ]

    return any(w in q for w in guest_words)



def query_mentions_hotel_date(query):
    q = _to_text(query)

    date_words = [
        "اليوم", "الليلة", "بكرة", "بكرا", "غدا", "غداً",
        "الأسبوع", "اسبوع", "أسبوع", "الشهر", "نهاية الأسبوع",
        "السبت", "الأحد", "الاحد", "الاثنين", "الثلاثاء",
        "الأربعاء", "الخميس", "الجمعة",
        "يناير", "فبراير", "مارس", "أبريل", "ابريل", "مايو",
        "يونيو", "يوليو", "أغسطس", "اغسطس", "سبتمبر",
        "أكتوبر", "اكتوبر", "نوفمبر", "ديسمبر"
    ]

    if any(w in q for w in date_words):
        return True

    # Numeric-looking dates: 10/15 or 10-15
    if re.search(r"\d{1,2}[-/]\d{1,2}", q):
        return True

    # Arabic-Indic numeric-looking dates: ١٠/١٥ or ١٠-١٥
    if re.search(r"[٠-٩]{1,2}[-/][٠-٩]{1,2}", q):
        return True

    # Date ranges like: من 10 ل 15 / من ٣ إلى ٧
    if re.search(r"(من|بين)\s*(\d+|[٠-٩]+)\s*(ل|إلى|الى|حتى|لحد)\s*(\d+|[٠-٩]+)", q):
        return True

    return False


def clean_iban_like_value(value):
    if value is None:
        return value

    s = str(value).strip()

    # Remove accidental float suffix
    if s.endswith(".0"):
        s = s[:-2]

    # Remove spaces inside IBAN-like strings only
    if re.match(r"^[A-Z]{2}\d+", s.replace(" ", "")):
        s = s.replace(" ", "")

    return s




import re


def normalize_guest_text(text):
    if not isinstance(text, str):
        return ""

    text = text.strip().lower()

    text = re.sub(r"[إأآٱ]", "ا", text)
    text = text.replace("ى", "ي")
    text = text.replace("ؤ", "و")
    text = text.replace("ئ", "ي")
    text = re.sub(r"[،,؛;.!؟?]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def detect_explicit_hotel_guests(query):
    text = normalize_guest_text(query)

    if not text:
        return None

    guest_patterns = [
        # شخص واحد
        (1, [
            r"\bلشخص\s+واحد\b",
            r"\bلشخصٍ?\b",
            r"\bشخص\s+واحد\b",
            r"\bضيف\s+واحد\b",
            r"\bلفرد\s+واحد\b",
        ]),

        # شخصان / شخصين / اثنان
        (2, [
            r"\bلشخصين\b",
            r"\bشخصين\b",
            r"\bشخصان\b",
        
            r"\bلاثنين\s+(?:اشخاص|ضيوف|افراد|ناس)\b",
            r"\bلاتنين\s+(?:اشخاص|ضيوف|افراد|ناس)\b",
            r"\bلثنين\s+(?:اشخاص|ضيوف|افراد|ناس)\b",
        
            r"\bاثنين\s+(?:اشخاص|ضيوف|افراد|ناس)\b",
            r"\bاتنين\s+(?:اشخاص|ضيوف|افراد|ناس)\b",
        
            r"\bضيفين\b",
            r"\bضيفان\b",
            r"\bلفردين\b",
        
            r"\bنحن\s+شخصين\b",
            r"\bاحنا\s+شخصين\b",
            r"\bعددنا\s+شخصين\b",
        ]),

        # ثلاثة
        (3, [
            r"\bلثلاثه\s+(?:اشخاص|ضيوف)\b",
            r"\bثلاثه\s+(?:اشخاص|ضيوف)\b",
            r"\bثلاث\s+(?:اشخاص|ضيوف)\b",
            r"\bلثلاثه\b",
            r"\b3\s*(?:اشخاص|ضيوف)\b",
        ]),

        # أربعة
        (4, [
            r"\بلاربعه\s+(?:اشخاص|ضيوف)\b",
            r"\باربعه\s+(?:اشخاص|ضيوف)\b",
            r"\باربع\s+(?:اشخاص|ضيوف)\b",
            r"\بلاربعه\b",
            r"\b4\s*(?:اشخاص|ضيوف)\b",
        ]),

        # خمسة
        (5, [
            r"\بلخمسه\s+(?:اشخاص|ضيوف)\b",
            r"\بخمسه\s+(?:اشخاص|ضيوف)\b",
            r"\بخمس\s+(?:اشخاص|ضيوف)\b",
            r"\بلخمسه\b",
            r"\b5\s*(?:اشخاص|ضيوف)\b",
        ]),

        # ستة
        (6, [
            r"\بلسته\s+(?:اشخاص|ضيوف)\b",
            r"\بسته\s+(?:اشخاص|ضيوف)\b",
            r"\بست\s+(?:اشخاص|ضيوف)\b",
            r"\بلسته\b",
            r"\b6\s*(?:اشخاص|ضيوف)\b",
        ]),
    ]

    for guests, patterns in guest_patterns:
        if any(re.search(pattern, text) for pattern in patterns):
            return guests

    return None

def query_mentions_guests(query):
    text = normalize_guest_text(query)

    guest_words = [
        "شخص",
        "شخصين",
        "اشخاص",
        "ضيف",
        "ضيفين",
        "ضيوف",
        "فرد",
        "فردين",
        "بالغ",
        "بالغين",
        "نحن",
        "احنا",
        "عددنا",
    ]

    return any(word in text for word in guest_words)

import re


def detect_weather_days(query):
    """
    Detects only very safe explicit weather-day expressions.

    Returns:
        2    when the query clearly refers to today and tomorrow
        None when no safe rule matches
    """
    if not isinstance(query, str):
        return None

    text = query.strip().lower()

    # توحيد بسيط للحروف والمسافات
    text = re.sub(r"[إأآٱ]", "ا", text)
    text = text.replace("ة", "ه")
    text = text.replace("ى", "ي")
    text = re.sub(r"[،,؛;.!؟?]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    two_day_patterns = [
        # اليوم + بكرة
        r"\bاليوم\s*(?:و|مع|و\s*)?\s*بكره\b",
        r"\bاليوم\s*(?:و|مع|و\s*)?\s*بكرا\b",
        r"\bاليوم\s*(?:و|مع|و\s*)?\s*غدا\b",
        r"\bاليوم\s*(?:و|مع|و\s*)?\s*الغد\b",

        # النهارده + بكرة
        r"\bالنهارده\s*(?:و|مع|و\s*)?\s*بكره\b",
        r"\bالنهارده\s*(?:و|مع|و\s*)?\s*بكرا\b",
        r"\bالنهارده\s*(?:و|مع|و\s*)?\s*غدا\b",

        # النهاردة + بكرة
        # بعد التطبيع "النهاردة" تصبح "النهارده"
        r"\bالنهارده\s*(?:و|مع|و\s*)?\s*بكره\b",
        r"\bالنهارده\s*(?:و|مع|و\s*)?\s*بكرا\b",

        # النهارة + بكرة
        r"\bالنهاره\s*(?:و|مع|و\s*)?\s*بكره\b",
        r"\bالنهاره\s*(?:و|مع|و\s*)?\s*بكرا\b",
        r"\bالنهاره\s*(?:و|مع|و\s*)?\s*غدا\b",

        # هالنهار + بكرة
        r"\bهالنهار\s*(?:و|مع|و\s*)?\s*بكره\b",
        r"\bهالنهار\s*(?:و|مع|و\s*)?\s*بكرا\b",

        # اليوم إلى بكرة
        r"\bاليوم\s+(?:الى|لبكره|لبكرا|لحد\s+بكره|لحد\s+بكرا)\b",
        r"\bالنهارده\s+(?:الى|لبكره|لبكرا|لحد\s+بكره|لحد\s+بكرا)\b",
        r"\bالنهاره\s+(?:الى|لبكره|لبكرا|لحد\s+بكره|لحد\s+بكرا)\b",

        # من اليوم إلى بكرة
        r"\bمن\s+اليوم\s+(?:الى|لبكره|لبكرا|لحد\s+بكره|لحد\s+بكرا)\b",
        r"\bمن\s+النهارده\s+(?:الى|لبكره|لبكرا|لحد\s+بكره|لحد\s+بكرا)\b",
        r"\bمن\s+النهاره\s+(?:الى|لبكره|لبكرا|لحد\s+بكره|لحد\s+بكرا)\b",

        # بكرة وبعد بكرة = يومان
        r"\bبكره\s*(?:و|مع)\s*بعد\s+بكره\b",
        r"\bبكرا\s*(?:و|مع)\s*بعد\s+بكرا\b",
        r"\bغدا\s*(?:و|مع)\s*بعد\s+غد\b",
    ]

    if any(re.search(pattern, text) for pattern in two_day_patterns):
        return 2

    return None

def query_mentions_today_only(query):
    """
    True إذا كان السؤال يطلب طقس اليوم فقط.

    لا تعتبره اليوم فقط إذا ظهر معه يوم آخر مثل:
    اليوم وبكرة
    اليوم والغد
    النهاردة وبكرة
    اليوم وبعد بكرة
    """

    if not isinstance(query, str):
        return False

    text = query.strip().lower()

    # توحيد الحروف
    text = re.sub(r"[إأآٱ]", "ا", text)
    text = text.replace("ة", "ه")
    text = text.replace("ى", "ي")
    text = re.sub(r"[،,؛;.!؟?]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    today_patterns = [
        r"\bاليوم\b",
        r"\bالنهارده\b",
        r"\bالنهاره\b",
        r"\bهالنهار\b",
    ]

    has_today = any(
        re.search(pattern, text)
        for pattern in today_patterns
    )

    if not has_today:
        return False

    # وجود أي يوم إضافي يعني أنها ليست "اليوم فقط"
    other_day_patterns = [
        r"\bبكره\b",
        r"\bبكرا\b",
        r"\bغدا\b",
        r"\bالغد\b",
        r"\bبعد\s+بكره\b",
        r"\bبعد\s+بكرا\b",
        r"\bبعد\s+غد\b",
        r"\bغدًا\b",
    ]

    has_another_day = any(
        re.search(pattern, text)
        for pattern in other_day_patterns
    )

    return not has_another_day

import re


def normalize_arabic_number_text(text):
    text = str(text or "").lower()
    text = re.sub(r"[إأآٱ]", "ا", text)
    text = text.replace("ى", "ي")
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def detect_explicit_currency_amount(query):
    q = normalize_arabic_number_text(query)

    # نص / نصف مليون
    if re.search(r"\b(?:نص|نصف)\s+مليون\b", q):
        return 500_000.0

    # مليون ونص / مليون ونصف
    if re.search(r"\bمليون\s+و?(?:نص|نصف)\b", q):
        return 1_500_000.0

    # مليون ومية ألف
    if re.search(
        r"\bمليون\s+و(?:ميه|مئه|مائه)\s+الف\b",
        q,
    ):
        return 1_100_000.0

    # مليون ومية
    if re.search(
        r"\bمليون\s+و(?:ميه|مئه|مائه)\b",
        q,
    ):
        return 1_000_100.0

    # ألف ومية
    if re.search(
        r"\bالف\s+و(?:ميه|مئه|مائه)\b",
        q,
    ):
        return 1_100.0

    # مية فقط
    if re.search(r"\b(?:ميه|مئه|مائه)\b", q):
        return 100.0

    return None

import re


def query_contains_digit(query):
    """
    يكتشف الأرقام الإنجليزية والعربية والفارسية.
    أمثلة:
    123
    ١٢٣
    ۱۲۳
    """
    if not isinstance(query, str):
        return False

    return bool(
        re.search(
            r"[0-9٠-٩۰-۹]",
            query
        )
    )


def keep_text_after_translation_separator(value):
    """
    إذا كان النص المتوقع يحتوي على:
    : أو ： أو ؟ أو ?
    ويأتي بعدها نص غير فارغ، نحتفظ بما بعدها فقط.

    لا يغيّر السؤال الذي تنتهي ترجمته بعلامة استفهام؛
    لأنه يشترط وجود نص بعد العلامة.
    """
    if not isinstance(value, str):
        return value

    value = value.strip()

    match = re.match(
        r"^.*[:：؟?]\s*(\S.*)$",
        value,
        flags=re.DOTALL
    )

    if match:
        cleaned_value = match.group(1).strip()

        if cleaned_value:
            return cleaned_value

    return value

def strip_outer_quotes(text):
    """
    يحذف علامات الاقتباس الخارجية فقط،
    ولا يغيّر علامات الاقتباس الموجودة داخل النص.
    """
    if not isinstance(text, str):
        return text

    text = text.strip()

    quote_pairs = [
        ("'", "'"),
        ('"', '"'),
        ("“", "”"),
        ("«", "»"),
        ("‘", "’"),
    ]

    changed = True

    while changed and len(text) >= 2:
        changed = False

        for left_quote, right_quote in quote_pairs:
            if text.startswith(left_quote) and text.endswith(right_quote):
                text = text[
                    len(left_quote):
                    len(text) - len(right_quote)
                ].strip()

                changed = True
                break

    return text


def extract_translation_text_from_query(query):
    """
    استخراج عام للنص المطلوب ترجمته من الـquery.

    الأولوية:
    1. النص المقتبس.
    2. النص بعد النقطتين.
    3. النص بعد علامة الاستفهام، بشرط وجود نص بعدها.
    """

    if not isinstance(query, str):
        return None

    text = query.strip()

    if not text:
        return None

    # ========================================================
    # 1. النص بين علامات الاقتباس
    #
    # مثال:
    # ترجم هذه العبارة: 'مرحبا، كيف حالك؟'
    #
    # نأخذ النص كاملًا حتى لو كان يحتوي ؟ أو :
    # ========================================================

    quote_patterns = [
        r"'([^']+)'",
        r'"([^"]+)"',
        r"“([^”]+)”",
        r"«([^»]+)»",
        r"‘([^’]+)’",
    ]

    quoted_candidates = []

    for pattern in quote_patterns:
        matches = re.findall(pattern, text)

        for match in matches:
            candidate = match.strip()

            if candidate:
                quoted_candidates.append(candidate)

    if quoted_candidates:
        # غالبًا آخر نص مقتبس هو النص المطلوب ترجمته
        return strip_outer_quotes(quoted_candidates[-1])

    # ========================================================
    # 2. النص بعد آخر نقطتين
    #
    # نستخدم آخر : حتى لو ظهر قبلها شرح طويل.
    #
    # مثال:
    # ترجم العبارة التالية إلى الإنجليزية: الجو جميل اليوم
    # ========================================================

    colon_matches = list(re.finditer(r"[:：]", text))

    if colon_matches:
        last_colon = colon_matches[-1]
        candidate = text[last_colon.end():].strip()
        candidate = strip_outer_quotes(candidate)

        if candidate:
            return candidate

    # ========================================================
    # 3. النص بعد علامة الاستفهام
    #
    # مثال:
    # ممكن تترجمها للإنجليزي؟ حبك مثل البحر ما ينتهي
    #
    # لا نستخدم ؟ إذا كانت آخر الجملة.
    # ========================================================

    question_marks = list(re.finditer(r"[؟?]", text))

    for question_mark in question_marks:
        candidate = text[question_mark.end():].strip()
        candidate = strip_outer_quotes(candidate)

        # لا بد أن يكون هناك نص حقيقي بعد علامة الاستفهام
        if candidate and re.search(r"[\w\u0600-\u06FF]", candidate):
            return candidate

    return None


import re
import unicodedata


ARABIC_DIGITS_MAP = str.maketrans(
    "٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹",
    "01234567890123456789",
)


HOTEL_MONTH_NAMES = (
    # الشهور الشائعة
    r"يناير|فبراير|مارس|ابريل|مايو|يونيو|يوليو|"
    r"اغسطس|سبتمبر|اكتوبر|نوفمبر|ديسمبر|"

    # بلاد الشام
    r"كانون\s+الثاني|شباط|اذار|نيسان|ايار|"
    r"حزيران|تموز|اب|ايلول|"
    r"تشرين\s+الاول|تشرين\s+الثاني|"

    # المغرب العربي
    r"جانفي|فيفري|افريل|جوان|جويليه|جويلية|"
    r"اوت|شتنبر|نونبر|دجنبر|"

    # الإنجليزية
    r"january|february|march|april|may|june|"
    r"july|august|september|october|november|december"
)


def normalize_date_query_text(text):
    text = str(text or "")
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(ARABIC_DIGITS_MAP)
    text = text.lower()

    # إزالة التشكيل والتطويل
    text = re.sub(
        r"[\u064B-\u065F\u0670\u0640]",
        "",
        text,
    )

    # توحيد الحروف
    text = re.sub(r"[إأآٱ]", "ا", text)
    text = text.replace("ى", "ي")
    text = text.replace("ة", "ه")

    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_hotel_date_range_from_query(query):
    text = normalize_date_query_text(query)

    if not text:
        return None

    separator = (
        r"(?:"
        r"الى|الي|حتي|حتى|لحد|لغاية|لغايه|لين|ل|-"
        r")"
    )

    # أمثلة:
    # من 2 حتى 7 نونبر
    # من 1 إلى 5 سبتمبر
    # من يوم 5 لغاية يوم 12 نوفمبر
    # بين 10 و15 ديسمبر
    shared_month_patterns = [
        (
            rf"\b(?:من|بين)\s*"
            rf"(?:تاريخ\s*|يوم\s*)?"
            rf"(?P<check_in_day>\d{{1,2}})\s*"
            rf"{separator}\s*"
            rf"(?:تاريخ\s*|يوم\s*)?"
            rf"(?P<check_out_day>\d{{1,2}})\s+"
            rf"(?P<month>{HOTEL_MONTH_NAMES})"
            rf"(?:\s+(?P<year>\d{{4}}))?"
        ),
        (
            rf"\bبين\s*"
            rf"(?:تاريخ\s*|يوم\s*)?"
            rf"(?P<check_in_day>\d{{1,2}})\s*"
            rf"و\s*"
            rf"(?:تاريخ\s*|يوم\s*)?"
            rf"(?P<check_out_day>\d{{1,2}})\s+"
            rf"(?P<month>{HOTEL_MONTH_NAMES})"
            rf"(?:\s+(?P<year>\d{{4}}))?"
        ),
    ]

    for pattern in shared_month_patterns:
        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE,
        )

        if match:
            check_in_day = match.group("check_in_day")
            check_out_day = match.group("check_out_day")
            month = match.group("month")
            year = match.group("year")

            check_in = f"{check_in_day} {month}"
            check_out = f"{check_out_day} {month}"

            if year:
                check_in = f"{check_in} {year}"
                check_out = f"{check_out} {year}"

            return {
                "check_in": check_in,
                "check_out": check_out,
            }

    # كل تاريخ مكتوب بشهر مستقل:
    # من 28 نوفمبر إلى 3 ديسمبر
    explicit_date = (
        rf"\d{{1,2}}\s+(?:{HOTEL_MONTH_NAMES})"
        rf"(?:\s+\d{{4}})?"
    )

    separate_month_pattern = (
        rf"\bمن\s*"
        rf"(?P<check_in>{explicit_date})\s*"
        rf"{separator}\s*"
        rf"(?P<check_out>{explicit_date})"
    )

    match = re.search(
        separate_month_pattern,
        text,
        flags=re.IGNORECASE,
    )

    if match:
        return {
            "check_in": match.group("check_in").strip(),
            "check_out": match.group("check_out").strip(),
        }

    # تواريخ رقمية:
    # من 20/6 إلى 25/6
    numeric_date = (
        r"\d{1,2}\s*[-/.]\s*\d{1,2}"
        r"(?:\s*[-/.]\s*\d{2,4})?"
    )

    numeric_range_pattern = (
        rf"\bمن\s*"
        rf"(?P<check_in>{numeric_date})\s*"
        rf"{separator}\s*"
        rf"(?P<check_out>{numeric_date})"
    )

    match = re.search(
        numeric_range_pattern,
        text,
        flags=re.IGNORECASE,
    )

    if match:
        return {
            "check_in": match.group("check_in").strip(),
            "check_out": match.group("check_out").strip(),
        }

    return None


def apply_hotel_query_date_format_rule(query, args):
    if not isinstance(args, dict):
        args = {}

    query_dates = extract_hotel_date_range_from_query(query)

    # لا يوجد مدى تاريخ واضح في النص:
    # لا نخترع أي قيمة ولا نغير ناتج المودل.
    if query_dates is None:
        return args

    fixed_args = dict(args)

    # عند وجود مدى صريح، نستخدم النص نفسه كمصدر موثوق،
    # سواء أخرج المودل الحقلين أم لم يخرجهما.
    fixed_args["check_in"] = query_dates["check_in"]
    fixed_args["check_out"] = query_dates["check_out"]

    return fixed_args

LANGUAGE_PATTERNS = [
    # English
    (r"(?:الانجليزيه|الانجليزي|الانكليزيه|الانكليزي|انجليزي|انكليزي|english)", "en"),

    # Arabic
    (r"(?:العربيه|العربي|عربي|العربي الفصيح|arabic)", "ar"),

    # French
    (r"(?:الفرنسيه|الفرنسي|الفرنساويه|الفرنساوي|فرنسي|فرنساوي|فرانساويه|فرانساوي|french)", "fr"),

    # German
    (r"(?:الالمانيه|الالماني|الماني|german)", "de"),

    # Spanish
    (r"(?:الاسبانيه|الاسباني|اسباني|spanish)", "es"),

    # Italian
    (r"(?:الايطاليه|الايطالي|ايطالي|italian)", "it"),

    # Persian
    (r"(?:الفارسيه|الفارسي|فارسي|persian|farsi)", "fa"),

    # Korean
    (r"(?:الكوريه|الكوري|كوري|korean)", "ko"),

    # Turkish
    (r"(?:التركيه|التركي|تركي|turkish)", "tr"),

    # Russian
    (r"(?:الروسيه|الروسي|روسي|russian)", "ru"),

    # Chinese
    (r"(?:الصينيه|الصيني|صيني|chinese)", "zh"),

    # Japanese
    (r"(?:اليابانيه|الياباني|ياباني|japanese)", "ja"),
]

def normalize_language_text(text):
    text = str(text).lower()
    text = re.sub(r"[\u064B-\u065F\u0670\u0640]", "", text)
    text = re.sub(r"[إأآٱ]", "ا", text)
    text = text.replace("ة", "ه")
    return text

def detect_explicit_translation_language(query):
    if not isinstance(query, str):
        return None

    text = normalize_language_text(query)

    for pattern, code in LANGUAGE_PATTERNS:
        if re.search(pattern, text):
            return code

    return None


In [32]:
# ============================================================
# Arabic normalization for safe rules
# ============================================================

def normalize_rule_text(text):
    text = str(text or "").strip().lower()

    text = (
        text.replace("أ", "ا")
            .replace("إ", "ا")
            .replace("آ", "ا")
            .replace("ٱ", "ا")
            .replace("ى", "ي")
            .replace("ة", "ه")
            .replace("ـ", "")
    )

    # Remove Arabic diacritics
    text = re.sub(
        r"[\u064B-\u065F\u0670]",
        "",
        text,
    )

    # Replace punctuation with spaces
    text = re.sub(
        r"[^\w\s]",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip()


def contains_any_phrase(text, phrases):
    normalized_text = normalize_rule_text(text)

    return any(
        normalize_rule_text(phrase) in normalized_text
        for phrase in phrases
    )


# ============================================================
# RULE 1: calculate_customs
# Remove "جهاز" only when it appears at the beginning
# of the predicted category.
#
# Examples:
# جهاز لابتوب        -> لابتوب
# جهاز كمبيوتر محمول -> كمبيوتر محمول
# جهاز تلفزيون       -> تلفزيون
#
# It will NOT modify:
# أجهزة إلكترونية
# إكسسوارات جهاز
# ============================================================

def remove_leading_device_word(value):
    if value is None:
        return value

    original_value = str(value).strip()

    cleaned_value = re.sub(
        r"^\s*جهاز\s+",
        "",
        original_value,
        count=1,
    ).strip()

    return cleaned_value or original_value

# ============================================================
# RULE 2: calculate_end_of_service
#
# Priority:
# 1) Explicit resignation
# 2) Unfair / unlawful dismissal
# 3) Economic layoff / redundancy
# 4) Disciplinary termination
# 5) End or expiry of contract
# 6) Explicit dismissal
# 7) Ambiguous wording -> remove termination_type
# ============================================================

RESIGNATION_PHRASES = [
    "استقلت",
    "استقالة",
    "استقالتي",
    "قدمت استقالتي",
    "قدمت الاستقالة",
    "قدمت استقالة",
    "تقدمت باستقالتي",
    "تقدمت باستقالة",
    "استقالة مني",
    "تركت العمل بإرادتي",
    "تركت العمل بارادتي",
    "أنهيت عملي بإرادتي",
    "انهيت عملي بارادتي",
    "إنهاء طوعي",
    "انهاء طوعي",
    "تسريح طوعي",
]


UNFAIR_DISMISSAL_PHRASES = [
    "فصل غير مشروع",
    "الفصل غير مشروع",
    "فصل تعسفي",
    "الفصل التعسفي",
    "إنهاء غير مشروع",
    "انهاء غير مشروع",
    "إنهاء خدمتي بشكل غير مشروع",
    "انهاء خدمتي بشكل غير مشروع",
    "فصل غير نظامي",
    "الفصل غير نظامي",
    "إنهاء غير نظامي",
    "انهاء غير نظامي",
]


ECONOMIC_PHRASES = [
    "تم الاستغناء عني",
    "استغنوا عني",
    "استغنت الشركة عني",
    "تم الاستغناء عن خدماتي",
    "الاستغناء عن خدماتي",
    "تقليص العمالة",
    "تقليص عدد الموظفين",
    "تخفيض العمالة",
    "تخفيض عدد الموظفين",
    "خفض عدد الموظفين",
    "إعادة هيكلة",
    "اعادة هيكلة",
    "إلغاء الوظيفة",
    "الغاء الوظيفة",
    "إلغاء منصبي",
    "الغاء منصبي",
    "تسريح الموظفين",
    "تم تسريحي",
    "اتسرحت",
    "اتسرح",
    "سرحوني",
    "أسباب اقتصادية",
    "اسباب اقتصادية",
    "ظروف اقتصادية",
    "ظروف الشركة",
    "إفلاس الشركة",
    "افلاس الشركة",
    "إغلاق الشركة",
    "اغلاق الشركة",
]


DISCIPLINARY_PHRASES = [
    "بسبب مخالفة",
    "بسبب المخالفة",
    "مخالفة وظيفية",
    "مخالفة ادارية",
    "مخالفة إدارية",
    "بسبب الغياب",
    "غياب متكرر",
    "بسبب سوء السلوك",
    "سوء السلوك",
    "مخالفة انضباطية",
    "مخالفة تأديبية",
    "فصل تأديبي",
]


END_OF_CONTRACT_PHRASES = [
    "انتهى عقدي",
    "انتهى العقد",
    "انتهاء عقدي",
    "انتهاء العقد",
    "نهاية العقد",
    "انقضاء العقد",
    "انقضى العقد",
    "عدم تجديد العقد",
    "لم يتم تجديد العقد",
    "ما جددوا عقدي",
    "ما تم تجديد عقدي",
    "انتهت مدة العقد",
    "انتهاء مدة العقد",
    "انقضاء مدة العقد",
    "انتهت مدة عملي",
]


DISMISSAL_PHRASES = [
    "تم فصلي",
    "فصلوني",
    "فصلتني الشركة",
    "تم طردي",
    "طردوني",
    "طردتني الشركة",
    "أنا مفصول",
    "انا مفصول",
    "إقالة",
    "اقالة",
    "بإقالة",
    "باقالة",
    "بطلب من الشركة",
    "إنهاء الخدمة بطلب من الشركة",
    "انهاء الخدمة بطلب من الشركة",
    "غير طوعي",
]


AMBIGUOUS_TERMINATION_PHRASES = [
    "توقفت عن العمل",
    "تركت العمل",
    "انتهت خدمتي",
    "إنهاء خدمتي",
    "انهاء خدمتي",
    "تم إنهاء خدمتي",
    "تم انهاء خدمتي",
    "إنهاء خدمة",
    "انهاء خدمة",
    "إنهاء عادي",
    "انهاء عادي",
    "إنهاء غير عادي",
    "انهاء غير عادي",
    "إنهاء طارئ",
    "انهاء طارئ",
    "انتهت علاقتي الوظيفية",
    "انتهت وظيفتي",
    "لم أعد أعمل",
    "لم اعد اعمل",
]


def detect_termination_type_from_query(query):
    """
    Returns:
        "resignation"
        "unfair_dismissal"
        "economic"
        "disciplinary"
        "end_of_contract"
        "dismissal"
        "__REMOVE__"
        None

    None means:
    Do not change the model prediction.
    """

    normalized_query = normalize_rule_text(query)

    # 1. Explicit resignation
    # Must come before contract wording because:
    # "سبب إنهاء العقد استقالة" -> resignation
    if contains_any_phrase(
        normalized_query,
        RESIGNATION_PHRASES,
    ):
        return "resignation"

    # 2. Unfair / unlawful dismissal
    if contains_any_phrase(
        normalized_query,
        UNFAIR_DISMISSAL_PHRASES,
    ):
        return "unfair_dismissal"

    # 3. Economic layoff / redundancy
    if contains_any_phrase(
        normalized_query,
        ECONOMIC_PHRASES,
    ):
        return "economic"

    # 4. Disciplinary termination
    if contains_any_phrase(
        normalized_query,
        DISCIPLINARY_PHRASES,
    ):
        return "disciplinary"

    # 5. Actual end or expiry of contract
    if contains_any_phrase(
        normalized_query,
        END_OF_CONTRACT_PHRASES,
    ):
        return "end_of_contract"

    # 6. Explicit dismissal
    if contains_any_phrase(
        normalized_query,
        DISMISSAL_PHRASES,
    ):
        return "dismissal"

    # 7. Ambiguous expression
    # Remove only if no explicit reason was detected above.
    if contains_any_phrase(
        normalized_query,
        AMBIGUOUS_TERMINATION_PHRASES,
    ):
        return "__REMOVE__"

    # No safe evidence: retain model prediction
    return None




In [33]:
import re


ARABIC_MONTHS = [
    # الشهور الشائعة
    "يناير", "فبراير", "مارس", "ابريل", "أبريل", "مايو",
    "يونيو", "يوليو", "اغسطس", "أغسطس", "سبتمبر",
    "اكتوبر", "أكتوبر", "نوفمبر", "ديسمبر",

    # بلاد الشام
    "كانون الثاني", "شباط", "اذار", "آذار", "نيسان",
    "ايار", "أيار", "حزيران", "تموز", "اب", "آب",
    "ايلول", "أيلول", "تشرين الاول", "تشرين الأول",
    "تشرين الثاني",

    # المغرب العربي
    "جانفي", "فيفري", "افريل", "أفريل", "جوان",
    "جويلية", "اوت", "أوت", "شتنبر", "نونبر", "دجنبر"
]

ENGLISH_MONTHS = [
    "january", "february", "march", "april", "may", "june",
    "july", "august", "september", "october", "november",
    "december"
]


def normalize_date_text(text):
    text = str(text or "").lower()

    # إزالة التشكيل والتطويل
    text = re.sub(r"[\u064B-\u065F\u0670\u0640]", "", text)

    # توحيد الحروف العربية
    text = re.sub(r"[إأآٱ]", "ا", text)
    text = text.replace("ى", "ي")
    text = text.replace("ة", "ه")

    # تحويل الأرقام العربية والفارسية إلى إنجليزية
    text = text.translate(
        str.maketrans(
            "٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹",
            "01234567890123456789"
        )
    )

    text = re.sub(r"\s+", " ", text).strip()
    return text


NORMALIZED_MONTHS = sorted(
    {
        normalize_date_text(month)
        for month in ARABIC_MONTHS + ENGLISH_MONTHS
    },
    key=len,
    reverse=True
)



def contains_explicit_month(query):
    q = normalize_date_text(query)

    for month in NORMALIZED_MONTHS:
        # \w يدعم الحروف العربية تلقائيًا،
        # لكنه لا يعتبر علامات الترقيم مثل ؟ حروفًا.
        pattern = (
            r"(?<!\w)"
            + re.escape(month)
            + r"(?!\w)"
        )

        if re.search(pattern, q):
            return True

    return False


def query_mentions_any_date(query):
    """
    يرجع True فقط عند وجود دلالة فعلية على تاريخ أو موعد.
    """
    q = normalize_date_text(query)

    if not q:
        return False

    # اسم شهر صريح
    if contains_explicit_month(q):
        return True

    # تاريخ ISO مثل 2026-07-25
    if re.search(
        r"\b\d{4}\s*-\s*\d{1,2}\s*-\s*\d{1,2}\b",
        q
    ):
        return True

    # تاريخ رقمي مثل 20/6 أو 20-06-2026
    if re.search(
        r"\b\d{1,2}\s*[-/.]\s*\d{1,2}"
        r"(?:\s*[-/.]\s*\d{2,4})?\b",
        q
    ):
        return True

    # مدى أيام:
    # من 2 حتى 7
    # من يوم 5 لغاية يوم 12
    # من 1 إلى 5
    # بين 10 و15
    range_patterns = [
        (
            r"\b(?:من|بين)\s*"
            r"(?:يوم\s*)?\d{1,2}\s*"
            r"(?:الي|الى|ل|لحد|لغاية|لغايه|حتى|-|و)\s*"
            r"(?:يوم\s*)?\d{1,2}\b"
        ),
        (
            r"\b\d{1,2}\s*"
            r"(?:الي|الى|لحد|لغاية|لغايه|حتى|-)\s*"
            r"\d{1,2}\b"
        )
    ]

    if any(re.search(pattern, q) for pattern in range_patterns):
        return True

    # يوم محدد
    if re.search(
        r"\b(?:يوم|بتاريخ|تاريخ)\s+\d{1,2}\b",
        q
    ):
        return True

    # تعبيرات زمنية نسبية
    relative_patterns = [
        r"\bاليوم\b",
        r"\bالليله\b",
        r"\bالنهارده\b",
        r"\bبكره\b",
        r"\bبكرا\b",
        r"\bغدا\b",
        r"\bبعد غد\b",
        r"\bبعد بكره\b",

        r"\bهذا الاسبوع\b",
        r"\bالاسبوع الحالي\b",
        r"\bالاسبوع ده\b",
        r"\bالاسبوع الجاي\b",
        r"\bالاسبوع القادم\b",
        r"\bالاسبوع المقبل\b",
        r"\bنهايه الاسبوع\b",

        r"\bهذا الشهر\b",
        r"\bالشهر الحالي\b",
        r"\bالشهر ده\b",
        r"\bالشهر الجاي\b",
        r"\bالشهر القادم\b",
        r"\bالشهر المقبل\b",
        r"\bنهايه الشهر\b",

        r"\bالعيد\b",
        r"\bعيد الفطر\b",
        r"\bعيد الاضحي\b",

        r"\bالاحد\b",
        r"\bالاثنين\b",
        r"\bالثلاثاء\b",
        r"\bالاربعاء\b",
        r"\bالخميس\b",
        r"\bالجمعه\b",
        r"\bالسبت\b",

        r"\bsunday\b",
        r"\bmonday\b",
        r"\btuesday\b",
        r"\bwednesday\b",
        r"\bthursday\b",
        r"\bfriday\b",
        r"\bsaturday\b"
    ]

    return any(
        re.search(pattern, q)
        for pattern in relative_patterns
    )

In [34]:
# ============================================================
# UNIFIED SAFE RULES — GENERAL + SPECIALIST
# يعمل على final_results_df بعد دمج نتائج المودلين
# ============================================================

def apply_safe_rules(pred_args, tool_name, row):
    args = (
        copy.deepcopy(pred_args)
        if isinstance(pred_args, dict)
        else {}
    )

    query = str(row.get("query", "") or "")

    # ---------------- SPECIALIST 7 TOOLS ----------------

    if tool_name == "order_food":
        restaurant = args.get("restaurant")
        if isinstance(restaurant, str):
            restaurant = re.sub(
                r"^مطعم\s+",
                "",
                restaurant.strip(),
            )
            args["restaurant"] = restaurant

    if tool_name == "calculate_customs":
        if "category" in args:
            args["category"] = remove_leading_device_word(
                args["category"]
            )

    if tool_name == "calculate_end_of_service":
        detected_type = detect_termination_type_from_query(query)

        if detected_type == "__REMOVE__":
            args.pop("termination_type", None)

        elif (
            detected_type is not None
            and "termination_type" in args
        ):
            args["termination_type"] = detected_type

    # transfer_money موجود في النوتبوكين؛ نطبقه مرة واحدة فقط
    if tool_name == "transfer_money":
        if "recipient_iban" in args:
            args["recipient_iban"] = clean_iban_like_value(
                args["recipient_iban"]
            )

    # ---------------- GENERAL TOOLS ----------------

    if tool_name == "calculate_zakat":
        currency = str(
            args.get("currency", "")
        ).strip().lower()
    
        invalid_weight_currencies = {
            "gram",
            "grams",
            "g",
            "kg",
            "kilogram",
            "kilograms",
            "جرام",
            "غرام",
            "جرامات",
            "غرامات",
            "كيلو",
            "كيلوجرام",
            "كيلوغرام",
        }
    
        if currency in invalid_weight_currencies:
            args.pop("currency", None)
    
        zakat_type = str(
            args.get("type", "")
        ).strip().lower()
    
        if zakat_type == "crop":
            args["type"] = "crops"
    
    if tool_name == "search_umrah_packages":
            if not query_mentions_any_date(query):
                args.pop("departure_date", None)

    if tool_name == "translate_text":
        extracted_from_query = extract_translation_text_from_query(query)

        if extracted_from_query:
            args["text"] = extracted_from_query
        else:
            predicted_text = args.get("text")

            if isinstance(predicted_text, str):
                cleaned_text = keep_text_after_translation_separator(
                    predicted_text
                )
                cleaned_text = strip_outer_quotes(cleaned_text)

                if cleaned_text:
                    args["text"] = cleaned_text

        detected_language = detect_explicit_translation_language(query)

        if detected_language is not None:
            args["target_language"] = detected_language
        else:
            args["target_language"] = "en"

    if tool_name == "check_visa_status":
        if not query_contains_digit(query):
            args.pop("visa_number", None)

    if tool_name == "search_hotels":
        detected_guests = detect_explicit_hotel_guests(query)

        if detected_guests is not None:
            args["guests"] = int(detected_guests)

        elif (
            "guests" in args
            and not query_mentions_guest_count(query)
        ):
            args.pop("guests", None)

        if not query_mentions_any_date(query):
            args.pop("check_in", None)
            args.pop("check_out", None)
        else:
            args = apply_hotel_query_date_format_rule(
                query=query,
                args=args,
            )

    if tool_name == "get_weather":
        detected_days = detect_weather_days(query)
    
        if detected_days is not None:
            detected_days = int(detected_days)
    
            if detected_days > 0:
                args["days"] = detected_days
            else:
                args.pop("days", None)
    
        elif query_mentions_today_only(query):
            args.pop("days", None)
    
        else:
            current_days = args.get("days")
    
            try:
                if float(current_days) <= 0:
                    args.pop("days", None)
            except (TypeError, ValueError):
                pass

    if tool_name == "get_qibla_direction":
        city = args.get("city")

        if isinstance(city, str):
            city = re.sub(
                r"^\s*مدينة\s+",
                "",
                city,
            ).strip()

            if city:
                args["city"] = city

    if tool_name == "convert_currency":
        detected_amount = detect_explicit_currency_amount(query)

        if detected_amount is not None:
            args["amount"] = detected_amount

    args = clean_args(args)
    args = clean_prediction_args(args, tool_name)

    return args


In [35]:
def canonicalize_json_value(value):
    if isinstance(value, dict):
        return {
            key: canonicalize_json_value(value[key])
            for key in sorted(value)
        }

    if isinstance(value, list):
        return [
            canonicalize_json_value(item)
            for item in value
        ]

    return value

In [36]:
# ============================================================
# APPLY RULES TO THE COMPLETE PIPELINE OUTPUT
# يحافظ على نفس الصفوف ونفس ترتيب الـID
# ============================================================

has_gold_arguments = False

final_results_rules_df = final_results_df.copy()

final_results_rules_df["predicted_arguments_before_rules"] = (
    final_results_rules_df["predicted_arguments"].apply(
        lambda value: copy.deepcopy(value)
    )
)

final_results_rules_df["predicted_arguments"] = [
    apply_safe_rules(
        pred_args=row["predicted_arguments_before_rules"],
        tool_name=row["tool_called"],
        row=row,
    )
    for _, row in final_results_rules_df.iterrows()
]

final_results_rules_df["rules_changed"] = [
    canonicalize_json_value(before)
    != canonicalize_json_value(after)
    for before, after in zip(
        final_results_rules_df["predicted_arguments_before_rules"],
        final_results_rules_df["predicted_arguments"],
    )
]

assert len(final_results_rules_df) == len(final_results_df)
assert (
    final_results_rules_df["row_number"].tolist()
    == final_results_df["row_number"].tolist()
)
assert (
    final_results_rules_df["id"].tolist()
    == final_results_df["id"].tolist()
)

print("Rows after rules:", len(final_results_rules_df))
print(
    "Rows changed by rules:",
    int(final_results_rules_df["rules_changed"].sum()),
)

if has_gold_arguments:
    final_results_rules_df["tool_correct"] = (
        final_results_rules_df["tool_called"]
        == final_results_rules_df["gold_tool"]
    )

    final_results_rules_df["arguments_correct"] = [
        arguments_exact_match(gold, pred)
        for gold, pred in zip(
            final_results_rules_df["gold_arguments"],
            final_results_rules_df["predicted_arguments"],
        )
    ]

    final_results_rules_df["full_call_correct"] = (
        final_results_rules_df["tool_correct"]
        & final_results_rules_df["arguments_correct"]
    )

    before_correct = int(
        final_results_df["full_call_correct"].sum()
    )
    after_correct = int(
        final_results_rules_df["full_call_correct"].sum()
    )
    total_rows = len(final_results_rules_df)

    print("=" * 72)
    print("END-TO-END DEV RESULTS — BEFORE VS AFTER RULES")
    print("=" * 72)
    print(
        f"Before rules: {before_correct}/{total_rows} "
        f"= {before_correct / total_rows:.2%}"
    )
    print(
        f"After rules : {after_correct}/{total_rows} "
        f"= {after_correct / total_rows:.2%}"
    )
    print(
        f"Correct rows change: {after_correct - before_correct:+d}"
    )

    rules_summary_df = (
        final_results_rules_df
        .groupby("argument_model", dropna=False)
        .agg(
            rows=("id", "size"),
            changed=("rules_changed", "sum"),
            correct_after=("full_call_correct", "sum"),
        )
        .reset_index()
    )

    before_by_model = (
        final_results_df
        .groupby("argument_model", dropna=False)
        ["full_call_correct"]
        .sum()
        .rename("correct_before")
        .reset_index()
    )

    rules_summary_df = rules_summary_df.merge(
        before_by_model,
        on="argument_model",
        how="left",
    )

    rules_summary_df["accuracy_before"] = (
        rules_summary_df["correct_before"]
        / rules_summary_df["rows"]
    )
    rules_summary_df["accuracy_after"] = (
        rules_summary_df["correct_after"]
        / rules_summary_df["rows"]
    )

    display(rules_summary_df)

    changed_rows_df = final_results_rules_df[
        final_results_rules_df["rules_changed"]
    ][
        [
            "row_number",
            "id",
            "query",
            "tool_called",
            "argument_model",
            "gold_arguments",
            "predicted_arguments_before_rules",
            "predicted_arguments",
            "full_call_correct",
        ]
    ].copy()

    display(changed_rows_df.head(50))


Rows after rules: 545
Rows changed by rules: 24


In [37]:
# ============================================================
# SAVE FINAL OUTPUT AFTER SAFE RULES
# Final predictions after query-grounded post-processing
# ============================================================

FINAL_RULED_JSONL = (
    PIPELINE_OUTPUT_DIR / "final_predictions_after_rules.jsonl"
)

FINAL_RULED_CSV = (
    PIPELINE_OUTPUT_DIR / "final_predictions_after_rules.csv"
)

RULES_AUDIT_XLSX = (
    PIPELINE_OUTPUT_DIR / "rules_audit.xlsx"
)

submission_records_after_rules = []

for _, row in final_results_rules_df.iterrows():
    submission_records_after_rules.append({
        "id": row["id"],
        "tool_called": row["tool_called"],
        "arguments": clean_args(
            row["predicted_arguments"]
        ),
    })

assert len(submission_records_after_rules) == len(test_raw)

assert [
    record["id"]
    for record in submission_records_after_rules
] == test_df.sort_values(
    "row_number"
)["id"].tolist()

with open(
    FINAL_RULED_JSONL,
    "w",
    encoding="utf-8",
) as output_file:
    for record in submission_records_after_rules:
        output_file.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )

submission_after_rules_csv_df = pd.DataFrame({
    "id": [
        record["id"]
        for record in submission_records_after_rules
    ],
    "tool_called": [
        record["tool_called"]
        for record in submission_records_after_rules
    ],
    "arguments": [
        json.dumps(
            record["arguments"],
            ensure_ascii=False,
        )
        for record in submission_records_after_rules
    ],
})

submission_after_rules_csv_df.to_csv(
    FINAL_RULED_CSV,
    index=False,
    encoding="utf-8-sig",
)

if has_gold_arguments:
    audit_export_df = final_results_rules_df.copy()

    for column in [
        "gold_arguments",
        "predicted_arguments_before_rules",
        "predicted_arguments",
    ]:
        audit_export_df[column] = audit_export_df[column].apply(
            lambda value: json.dumps(
                value,
                ensure_ascii=False,
            )
        )

    with pd.ExcelWriter(
        RULES_AUDIT_XLSX,
        engine="openpyxl",
    ) as writer:
        audit_export_df.to_excel(
            writer,
            sheet_name="all_rows",
            index=False,
        )

        audit_export_df[
            audit_export_df["rules_changed"]
        ].to_excel(
            writer,
            sheet_name="changed_by_rules",
            index=False,
        )

print("=" * 72)
print("FINAL AFTER-RULES VALIDATION PASSED")
print("=" * 72)
print("Rows:", len(submission_records_after_rules))
print("Output:", FINAL_RULED_JSONL)
print("CSV:", FINAL_RULED_CSV)

if has_gold_arguments:
    print("Audit:", RULES_AUDIT_XLSX)


FINAL AFTER-RULES VALIDATION PASSED
Rows: 545
Output: /kaggle/working/aisa_final_pipeline/final_predictions_after_rules.jsonl
CSV: /kaggle/working/aisa_final_pipeline/final_predictions_after_rules.csv
